# M25a · Gaussienne multivariée, conditionnement et divergence KL — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m25a_gaussienne/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m25a_gaussienne/03_notebook.ipynb`, créée par `python tools/start_chapter.py M25a`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M25a.1 à M25a.9, M25a.16 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M25a.20 | La log-densité d'une loi normale multivariée, par Cholesky · `mvn_logpdf` | 🔨 | ★★ | 30 |
|  | M25a.21 | Tirer des vecteurs de loi N(μ, Σ) par Cholesky · `sample_mvn` | 🔨 | ★ | 20 |
|  | M25a.22 | Le complément de Schur d'un bloc, et la loi conditionnelle d'un vecteur gaussien · `schur_complement`, `condition_gaussian` | 🔨 | ★★ | 35 |
|  | M25a.23 | Le posterior d'une régression linéaire bayésienne · `linreg_posterior` | 🔨 | ★★ | 25 |
|  | M25a.24 | `scipy.stats.multivariate_normal`, `np.cov` et `scipy.integrate` : tes log-densités, le maximum de vraisemblance, et la KL de M25a.8 par une intégrale double | 📦 | ★ | 25 |
| B | M25a.25 | Prédire la part des tirages dans l'ellipsoïde de Mahalanobis de même niveau quand la dimension passe de 2 à 10, puis mesurer | 🔮 | ★ | 15 |
|  | M25a.26 | Conditionner par simulation : garder les tirages dont une mesure tombe près d'une valeur, et comparer à la loi conditionnelle exacte | 🔬 | ★★ | 25 |
|  | M25a.27 | Les ellipses de niveau d'une gaussienne en dimension 2, une tranche (la loi conditionnelle) et une projection (la loi marginale), selon la corrélation | 🎨 | ★ | 20 |
| C | M25a.28 | Chasse à l'erreur : un tirage par E L au lieu de E Lᵀ, ln det Σ sans le facteur 2, et une covariance conditionnelle qui dépend de la valeur observée | 🐛 | ★ | 20 |
|  | M25a.29 | Un test de propriété : la moyenne et la covariance empiriques de tes tirages, à une tolérance fixée par l'erreur type | 🛠️ | ★ | 20 |
| D | M25a.30 | Défi : reconstruire les températures manquantes le long d'une route par la loi conditionnelle d'une gaussienne, sous une erreur quadratique moyenne fixée | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25a")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : la densité d'un couple gaussien, des images linéaires et un tirage par Cholesky, des ellipses de niveau, un capteur estimé par ses voisins, la fusion de mesures, une régression bayésienne, des divergences KL, une route orthodromique, et un pont vers les tableaux en lignes.
- Écrire les cinq fonctions de `mymath/gaussian.py` (`mvn_logpdf`, `sample_mvn`, `schur_complement`, `condition_gaussian`, `linreg_posterior`) et les tester.
- Comparer ta librairie à SciPy, prédire et mesurer la masse d'un ellipsoïde en dimension croissante, conditionner par simulation, dessiner des ellipses, une tranche et une projection, corriger trois erreurs classiques, écrire un test statistique, et relever un défi.

**Rappel express.** $\ln\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma}) = -\frac12\big[d\ln(2\pi) + \ln\det\boldsymbol{\Sigma} + \Delta^2\big]$, $\Delta^2 = (\mathbf{x} - \boldsymbol{\mu})^\top\boldsymbol{\Sigma}^{-1}(\mathbf{x} - \boldsymbol{\mu}) = \lVert\mathbf{L}^{-1}(\mathbf{x} - \boldsymbol{\mu})\rVert^2$, $\ln\det\boldsymbol{\Sigma} = 2\sum_i\ln L_{ii}$. $\mathbf{A}\mathbf{x} + \mathbf{b} \sim \mathcal{N}(\mathbf{A}\boldsymbol{\mu} + \mathbf{b}, \mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^\top)$ ; $\boldsymbol{\mu} + \mathbf{L}\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{\mu}, \mathbf{L}\mathbf{L}^\top)$, en lignes `mean + E @ L.T`. Ellipse $\Delta = c$ : demi-axes $c\sqrt{\lambda_i}$, aire $\pi c^2\sqrt{\det\boldsymbol{\Sigma}}$, masse $1 - e^{-c^2/2}$ en dimension 2 ; $\Delta^2 \sim \chi^2_d$. $\mathbf{x}_a \mid \mathbf{x}_b \sim \mathcal{N}\big(\boldsymbol{\mu}_a + \boldsymbol{\Sigma}_{ab}\boldsymbol{\Sigma}_{bb}^{-1}(\mathbf{x}_b - \boldsymbol{\mu}_b),\ \boldsymbol{\Sigma}_{aa} - \boldsymbol{\Sigma}_{ab}\boldsymbol{\Sigma}_{bb}^{-1}\boldsymbol{\Sigma}_{ba}\big)$ ; $\boldsymbol{\Sigma}_{a \mid b} = \boldsymbol{\Lambda}_{aa}^{-1}$. Bayes gaussien : les précisions s'additionnent ; $\mathbf{S}_N = (\mathbf{S}_0^{-1} + \sigma^{-2}\boldsymbol{\Phi}^\top\boldsymbol{\Phi})^{-1}$, $\mathbf{m}_N = \mathbf{S}_N(\mathbf{S}_0^{-1}\mathbf{m}_0 + \sigma^{-2}\boldsymbol{\Phi}^\top\mathbf{y})$. $\hat{\boldsymbol{\Sigma}} = \mathbf{S}/n$ (`np.cov(X, rowvar=False, bias=True)`). $\mathrm{KL}(p \,\|\, q) = \mathbb{E}_p[\ln p - \ln q] \ge 0$ ; entre deux gaussiennes, $\frac12\big[\operatorname{tr}(\boldsymbol{\Sigma}_1^{-1}\boldsymbol{\Sigma}_0) + \boldsymbol{\delta}^\top\boldsymbol{\Sigma}_1^{-1}\boldsymbol{\delta} - d + \ln\frac{\det\boldsymbol{\Sigma}_1}{\det\boldsymbol{\Sigma}_0}\big]$. Slerp : $\frac{\sin((1 - t)\Omega)}{\sin\Omega}\mathbf{u} + \frac{\sin(t\Omega)}{\sin\Omega}\mathbf{v}$.

## Partie 0 · Vérifier tes exercices papier (✏️ M25a.1 à M25a.9, 🔗 M25a.16)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]` ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]` ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit ;
- une valeur exacte entre guillemets, écrite avec des entiers, des fractions, `sqrt`, `pi`, `ln` et `exp`, comme `"sqrt(5)/3"` ou `"2*ln(7)"` ;
- une expression entre guillemets, comme `"x*exp(-x)"` ;
- une **forme** comme un tuple, `(7, 2)`, ou comme un texte, `"(7, 2)"` ;
- `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex M25a.1 — La densité d'un couple gaussien (température et humidité d'une serre) en deux points**

In [ ]:
# Ex M25a.1: the values you found on paper
answer_M25a_1a = ...  # a) an integer
answer_M25a_1b = ...  # b) a 2 × 2 matrix (its coefficients are exact decimals)
answer_M25a_1c = ...  # c) a 2 × 2 matrix
answer_M25a_1d = ...  # d) a list of 2 numbers
answer_M25a_1e = ...  # e) an exact value, as text, with sqrt, pi, ln, exp and fractions
answer_M25a_1f = ...  # f) a list of 2 numbers (2 decimals)
answer_M25a_1g = ...  # g) a number (4 decimals)
answer_M25a_1h = ...  # h) an exact value, as text, with sqrt, pi, ln, exp and fractions
answer_M25a_1i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M25a_1a, answer_M25a_1b, answer_M25a_1c, answer_M25a_1d, answer_M25a_1e, answer_M25a_1f, answer_M25a_1g, answer_M25a_1h, answer_M25a_1i]):
    wbm.check(f"M25a.1{letter}", answer)

**Ex M25a.2 — Trois mesures corrélées : une somme pondérée, une différence, un tirage μ + Lε**

In [ ]:
# Ex M25a.2: the values you found on paper
answer_M25a_2a = ...  # a) a 3 × 3 matrix
answer_M25a_2b = ...  # b) an integer
answer_M25a_2c = ...  # c) a list of 2 numbers
answer_M25a_2d = ...  # d) a list of 2 numbers
answer_M25a_2e = ...  # e) a 2 × 2 matrix
answer_M25a_2f = ...  # f) a list of 3 numbers
answer_M25a_2g = ...  # g) a 3 × 3 matrix
answer_M25a_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M25a_2a, answer_M25a_2b, answer_M25a_2c, answer_M25a_2d, answer_M25a_2e, answer_M25a_2f, answer_M25a_2g, answer_M25a_2h]):
    wbm.check(f"M25a.2{letter}", answer)

**Ex M25a.3 — Les ellipses à 90 % et à 95 % des composantes du vent**

In [ ]:
# Ex M25a.3: the values you found on paper
answer_M25a_3a = ...  # a) a list of 2 numbers
answer_M25a_3b = ...  # b) an irreducible fraction, as text "p/q"
answer_M25a_3c = ...  # c) a list of 2 numbers (4 decimals)
answer_M25a_3d = ...  # d) a list of 2 numbers (4 decimals)
answer_M25a_3e = ...  # e) a list of 2 numbers (4 decimals)
answer_M25a_3f = ...  # f) a list of 2 numbers (2 decimals)
answer_M25a_3g = ...  # g) a number (4 decimals)
answer_M25a_3h = ...  # h) an irreducible fraction, as text "p/q"
answer_M25a_3i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M25a_3a, answer_M25a_3b, answer_M25a_3c, answer_M25a_3d, answer_M25a_3e, answer_M25a_3f, answer_M25a_3g, answer_M25a_3h, answer_M25a_3i]):
    wbm.check(f"M25a.3{letter}", answer)

**Ex M25a.4 — Un capteur en panne estimé par ses deux voisins : complément de Schur et loi conditionnelle**

In [ ]:
# Ex M25a.4: the values you found on paper
answer_M25a_4a = ...  # a) a 2 × 2 matrix (4 decimals)
answer_M25a_4b = ...  # b) a list of 2 numbers (4 decimals)
answer_M25a_4c = ...  # c) an irreducible fraction, as text "p/q"
answer_M25a_4d = ...  # d) a number (2 decimals)
answer_M25a_4e = ...  # e) a number (4 decimals)
answer_M25a_4f = ...  # f) a list of 2 numbers (3 decimals)
answer_M25a_4g = ...  # g) an integer
answer_M25a_4h = ...  # h) an irreducible fraction, as text "p/q"
answer_M25a_4i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M25a_4a, answer_M25a_4b, answer_M25a_4c, answer_M25a_4d, answer_M25a_4e, answer_M25a_4f, answer_M25a_4g, answer_M25a_4h, answer_M25a_4i]):
    wbm.check(f"M25a.4{letter}", answer)

**Ex M25a.5 — Fusionner trois mesures d'une même distance : des précisions qui s'additionnent**

In [ ]:
# Ex M25a.5: the values you found on paper
answer_M25a_5a = ...  # a) a list of 2 numbers (4 decimals)
answer_M25a_5b = ...  # b) a list of 2 numbers (4 decimals)
answer_M25a_5c = ...  # c) a list of 2 numbers (2 decimals)
answer_M25a_5d = ...  # d) True or False
answer_M25a_5e = ...  # e) an irreducible fraction, as text "p/q"
answer_M25a_5f = ...  # f) a number (4 decimals)
answer_M25a_5g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M25a_5a, answer_M25a_5b, answer_M25a_5c, answer_M25a_5d, answer_M25a_5e, answer_M25a_5f, answer_M25a_5g]):
    wbm.check(f"M25a.5{letter}", answer)

**Ex M25a.6 — Étalonner un thermomètre par la régression linéaire bayésienne**

In [ ]:
# Ex M25a.6: the values you found on paper
answer_M25a_6a = ...  # a) a 2 × 2 matrix
answer_M25a_6b = ...  # b) a 2 × 2 matrix
answer_M25a_6c = ...  # c) a number (4 decimals)
answer_M25a_6d = ...  # d) a 2 × 2 matrix (4 decimals)
answer_M25a_6e = ...  # e) a list of 2 numbers (4 decimals)
answer_M25a_6f = ...  # f) a list of 2 numbers (4 decimals)
answer_M25a_6g = ...  # g) a list of 2 numbers (4 decimals)
answer_M25a_6h = ...  # h) a number (4 decimals)
answer_M25a_6i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M25a_6a, answer_M25a_6b, answer_M25a_6c, answer_M25a_6d, answer_M25a_6e, answer_M25a_6f, answer_M25a_6g, answer_M25a_6h, answer_M25a_6i]):
    wbm.check(f"M25a.6{letter}", answer)

**Ex M25a.7 — La divergence KL continue entre deux lois exponentielles et entre deux lois uniformes**

In [ ]:
# Ex M25a.7: the values you found on paper
answer_M25a_7a = ...  # a) an expression in a and b, as text
answer_M25a_7b = ...  # b) a number (3 decimals)
answer_M25a_7c = ...  # c) a number (3 decimals)
answer_M25a_7d = ...  # d) a number (3 decimals)
answer_M25a_7e = ...  # e) True or False
answer_M25a_7f = ...  # f) a number (3 decimals)
answer_M25a_7g = ...  # g) a number (3 decimals)
answer_M25a_7h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M25a_7a, answer_M25a_7b, answer_M25a_7c, answer_M25a_7d, answer_M25a_7e, answer_M25a_7f, answer_M25a_7g, answer_M25a_7h]):
    wbm.check(f"M25a.7{letter}", answer)

**Ex M25a.8 — Deux réglages d'une machine de découpe : la KL entre deux gaussiennes dans les deux sens**

In [ ]:
# Ex M25a.8: the values you found on paper
answer_M25a_8a = ...  # a) an irreducible fraction, as text "p/q"
answer_M25a_8b = ...  # b) an irreducible fraction, as text "p/q"
answer_M25a_8c = ...  # c) an exact value, as text, with sqrt, pi, ln, exp and fractions
answer_M25a_8d = ...  # d) a number (4 decimals)
answer_M25a_8e = ...  # e) an irreducible fraction, as text "p/q"
answer_M25a_8f = ...  # f) an irreducible fraction, as text "p/q"
answer_M25a_8g = ...  # g) a number (4 decimals)
answer_M25a_8h = ...  # h) an irreducible fraction, as text "p/q"
answer_M25a_8i = ...  # i) a number (4 decimals)

for letter, answer in zip("abcdefghi", [answer_M25a_8a, answer_M25a_8b, answer_M25a_8c, answer_M25a_8d, answer_M25a_8e, answer_M25a_8f, answer_M25a_8g, answer_M25a_8h, answer_M25a_8i]):
    wbm.check(f"M25a.8{letter}", answer)

**Ex M25a.9 — La route orthodromique entre deux villes : angle au centre, slerp au tiers, corde**

In [ ]:
# Ex M25a.9: the values you found on paper
answer_M25a_9a = ...  # a) a list of 3 numbers (4 decimals)
answer_M25a_9b = ...  # b) a list of 3 numbers (4 decimals)
answer_M25a_9c = ...  # c) a number (4 decimals)
answer_M25a_9d = ...  # d) an integer
answer_M25a_9e = ...  # e) an integer
answer_M25a_9f = ...  # f) a list of 3 numbers (4 decimals)
answer_M25a_9g = ...  # g) a list of 2 numbers (2 decimals)
answer_M25a_9h = ...  # h) an integer
answer_M25a_9i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M25a_9a, answer_M25a_9b, answer_M25a_9c, answer_M25a_9d, answer_M25a_9e, answer_M25a_9f, answer_M25a_9g, answer_M25a_9h, answer_M25a_9i]):
    wbm.check(f"M25a.9{letter}", answer)

**Ex M25a.16 — Pont : des vecteurs rangés en lignes (n, d), tirés et évalués d'un coup**

In [ ]:
# Ex M25a.16: the values you found on paper
answer_M25a_16a = ...  # a) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M25a_16b = ...  # b) a 3 × 2 matrix
answer_M25a_16c = ...  # c) a list of 3 numbers
answer_M25a_16d = ...  # d) a list of 3 numbers (4 decimals)
answer_M25a_16e = ...  # e) a number (4 decimals)
answer_M25a_16f = ...  # f) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M25a_16g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M25a_16a, answer_M25a_16b, answer_M25a_16c, answer_M25a_16d, answer_M25a_16e, answer_M25a_16f, answer_M25a_16g]):
    wbm.check(f"M25a.16{letter}", answer)

## Partie A · Ta librairie : log-densité, tirages, conditionnement, régression bayésienne, et SciPy

Fiche §1, §2, §4 à §7. Les cinq fonctions de `mymath/gaussian.py`, leurs tests, et un usage de chacune ; puis SciPy et `np.cov` pour vérifier, estimer et intégrer.

In [ ]:
# Tools of the notebook (parts A to D)
import math
import numbers
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate, stats

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, pair, run_pytest_group, same, verdict

TEST_M25A = "tests/test_m25a_gaussian.py"            # the tests of mymath/gaussian.py


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        if isinstance(value, tuple):
            return "(" + ", ".join(str(np.round(np.asarray(v, dtype=float), digits).tolist()) for v in value) + ")"
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())
    except (TypeError, ValueError):
        return repr(value)[:120]


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def line_points(line):
    """The (x, y) points of a line of matplotlib, as two float arrays."""
    return np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)


def through(ax, xs, ys, rtol=1e-9):
    """True when one line of ax passes by all the points (xs, ys), in this order or in the reverse order."""
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    for line in ax.lines:
        x, y = line_points(line)
        for a, b in ((xs, ys), (xs[::-1], ys[::-1])):
            if (x.shape == a.shape and y.shape == b.shape and np.allclose(x, a, rtol=rtol, atol=1e-12)
                    and np.allclose(y, b, rtol=rtol, atol=1e-12)):
                return True
    return False


def horizontal(ax, level):
    """True when ax has a horizontal line at the height level (ax.axhline, or a line of constant height)."""
    return any(line_points(line)[1].size >= 2 and np.allclose(line_points(line)[1], level, rtol=1e-9, atol=1e-12)
               for line in ax.lines)


def has_points(ax, points):
    """True when every (x, y) of points is a point of a line (markers) or of a scatter of ax."""
    found = []
    for line in ax.lines:
        x, y = line_points(line)
        found += list(zip(x, y))
    for collection in ax.collections:
        offsets = np.asarray(collection.get_offsets(), dtype=float)
        if offsets.ndim == 2 and offsets.shape[1] == 2:
            found += [tuple(p) for p in offsets]
    return all(any(abs(fx - px) < 1e-9 and abs(fy - py) < 1e-9 for fx, fy in found) for px, py in points)

### Ex M25a.20 — La log-densité d'une loi normale multivariée, par Cholesky · `mvn_logpdf` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer la log-densité d'une loi normale multivariée par le facteur de Cholesky, puis voir qu'en grande dimension la densité sort des flottants et son logarithme non.  
**Prérequis :** fiche §1 ; M12c ; Ex M25a.1 et M25a.16 · **mymath :** `gaussian.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/gaussian.py`, lis la docstring du module puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.gaussian.<nom>` (par exemple `mymath.gaussian.mvn_logpdf(x, mean, cov)`), ou donne-leur un nom court (`mvn_logpdf = mymath.gaussian.mvn_logpdf`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute `import numpy as np` en tête du fichier (et `import numbers` ou `import math` si tu t'en sers). NumPy est permis, mais, de `np.linalg`, seulement `cholesky` et `solve` (et `np.linalg.LinAlgError` pour attraper un échec), ni `multivariate_normal` (ni celui de NumPy, ni la méthode d'un `Generator`), et aucune fonction de SciPy : les tests comparent tes résultats à ceux de `scipy.stats.multivariate_normal`, et un test lit ton module. Les indices d'un bloc se donnent par une **liste** (`[0, 2]`, jamais un entier nu), numérotés à partir de 0. Les nombres se rendent en `float` de Python, les vecteurs et les matrices en tableaux NumPy de flottants ; ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `mvn_logpdf(x, mean, cov)` (fiche §1), en suivant sa docstring : la log-densité de $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ en un point de forme $(d,)$, ou en $n$ points rangés en lignes $(n, d)$, par le facteur de Cholesky : $\ln\det\boldsymbol{\Sigma} = 2\sum_i\ln L_{ii}$ et $\Delta^2 = \lVert\mathbf{y}\rVert^2$, où $\mathbf{y}$ résout $\mathbf{L}\mathbf{y} = \mathbf{x} - \boldsymbol{\mu}$, jamais par `det` ni par une inverse. Un `float` pour un point, un tableau de $n$ flottants pour $n$ points ; `ValueError` pour les formes et les covariances que refuse la docstring (pas symétrique, pas définie positive).

**L'usage.** La cellule fournie donne la serre de M25a.1 (`MU_GREEN_20`, `COV_GREEN_20`, `XP_20`, `XQ_20`), le pont M25a.16 (`MU_16_20`, `COV_16_20`, `L_16_20`, `E_16_20`), et la loi de dimension 600 de moyenne nulle et de covariance $\Sigma_{ij} = 0{,}3^{\lvert i - j\rvert}$ (`COV_20`), avec `EPS_20` $= $ `np.random.default_rng(5920).standard_normal(600)`. Avec ta fonction :
- a) $\ln p(\mathbf{x}_P)$ et $\ln p(\mathbf{x}_Q)$ pour la serre, à comparer à ta réponse f) de M25a.1 (une liste de 2 nombres) ;
- b) les trois log-densités des lignes de $\mathbf{X} = \mathbf{1}\boldsymbol{\mu}^\top + \mathbf{E}\mathbf{L}^\top$ du 🔗 M25a.16, d'un seul appel avec un `x` de forme $(3, 2)$, à comparer à ta réponse d) (une liste de 3 nombres) ;
- c) en dimension 600, $\ln p(\mathbf{x})$ au point $\mathbf{x} = \mathbf{L}\boldsymbol{\varepsilon}$, où $\mathbf{L}$ = `np.linalg.cholesky(COV_20)` et $\boldsymbol{\varepsilon}$ = `EPS_20` (2 décimales dans ta copie) ;
- d) vrai ou faux : `np.exp` de cette valeur vaut exactement `0.0` ;
- e) $\ln\det\boldsymbol{\Sigma}$ par $2\sum_i\ln L_{ii}$, à comparer à la valeur exacte $599\ln(1 - 0{,}3^2)$ (2 décimales dans ta copie).

Ce qu'il faut observer : en grande dimension, la densité d'un point tout à fait ordinaire sort des flottants, son logarithme non ; et le déterminant, minuscule, se calcule par son logarithme, une somme de $d$ termes. Dans ta copie : pourquoi $\ln\det\boldsymbol{\Sigma} = 2\sum_i\ln L_{ii}$ ? Quel est l'ordre de grandeur de $\det\boldsymbol{\Sigma}$ en dimension 600 ?

In [ ]:
MU_GREEN_20 = np.array([21.0, 65.0])                       # the greenhouse of M25a.1: (T, H)
COV_GREEN_20 = np.array([[4.0, -6.0], [-6.0, 34.0]])
XP_20, XQ_20 = np.array([23.0, 68.0]), np.array([23.0, 60.0])
MU_16_20 = np.array([7.0, 4.0])                            # the bridge M25a.16
COV_16_20 = np.array([[4.0, -2.0], [-2.0, 10.0]])
L_16_20 = np.array([[2.0, 0.0], [-1.0, 3.0]])
E_16_20 = np.array([[1.0, 0.0], [0.5, -1.0], [-1.0, 2.0]])  # one draw of N(0, I) per row: X = 1 μᵀ + E Lᵀ
D_20 = 600
COV_20 = 0.3 ** np.abs(np.arange(D_20)[:, None] - np.arange(D_20)[None, :])    # Σ_ij = 0.3^|i − j|
EPS_20 = np.random.default_rng(5920).standard_normal(D_20)                      # x = L ε, L = np.linalg.cholesky(COV_20)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25a")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M25A, "test_mvn_logpdf_ or test_module_uses_")

**L'usage** : la serre, le pont en lignes, et une loi de dimension 600.

In [ ]:
answer_M25a_20a = answer_M25a_20b = answer_M25a_20c = answer_M25a_20d = answer_M25a_20e = ...
with wbm.attempt("M25a.20"):          # if you call mvn_logpdf here: ⏳ while it is not written
    answer_M25a_20a = ...  # a) a list of 2 numbers, as computed: ln p(x_P) and ln p(x_Q) for the greenhouse
    answer_M25a_20b = ...  # b) a list of 3 numbers, as computed: the log-densities of the 3 rows of X (M25a.16), one call
    answer_M25a_20c = ...  # c) a number, as computed: ln p(x) in dimension 600, at x = L EPS_20
    answer_M25a_20d = ...  # d) True or False: np.exp of c) is exactly 0.0
    answer_M25a_20e = ...  # e) a number, as computed: ln det Σ = 2 Σ_i ln L_ii in dimension 600

In [ ]:
for letter, answer in zip("abcde", [answer_M25a_20a, answer_M25a_20b, answer_M25a_20c, answer_M25a_20d,
                                       answer_M25a_20e]):
    wbm.check(f"M25a.20{letter}", answer, computed=True)

### Ex M25a.21 — Tirer des vecteurs de loi N(μ, Σ) par Cholesky · `sample_mvn` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le tirage de vecteurs gaussiens par Cholesky, puis vérifier leurs moments et les blanchir.  
**Prérequis :** fiche §2 ; M12c ; Ex M25a.2 · **mymath :** `gaussian.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M25a.20 : écris la fonction dans ta copie `mon_travail/mymath/gaussian.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `sample_mvn(mean, cov, n, seed=0)` (fiche §2), en suivant sa docstring : $n$ vecteurs de loi $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ rangés en lignes, avec **un seul** appel `rng.standard_normal((n, d))`, où `rng = np.random.default_rng(seed)`, puis `mean + E @ L.T` (la ligne $i$ est $\boldsymbol{\mu} + \mathbf{L}\boldsymbol{\varepsilon}_i$). Un tableau $(n, d)$ ; `ValueError` pour un `n` ou une graine qui ne sont pas des entiers convenables, et pour les covariances que refuse la docstring.

**L'usage.** La cellule fournie donne les épaisseurs de route de M25a.2 (`MU_ROAD_21`, `COV_ROAD_21`) et les coûts par millimètre (`COST_21`). Avec ta fonction, 20 000 tirages des épaisseurs (`seed=5921`, `sample_21`), puis :
- a) leur moyenne empirique (une liste de 3 nombres ; 3 décimales dans ta copie) et leur covariance empirique `np.cov(sample_21, rowvar=False, bias=True)` (`cov_21` ; 2 décimales dans ta copie), à comparer à $\boldsymbol{\mu}$ et $\boldsymbol{\Sigma}$ ;
- b) le coût $c = 2x_1 + x_2 + 3x_3$ de chaque tirage : sa moyenne et sa variance empiriques (`np.var` ; une liste de 2 nombres ; 2 décimales dans ta copie), à comparer à ta réponse c) de M25a.2 ;
- c) les tirages blanchis, $\mathbf{L}^{-1}(\mathbf{x} - \boldsymbol{\mu})$ pour chaque ligne (par `np.linalg.solve`) : leur covariance empirique (`bias=True` ; une matrice $3 \times 3$ ; 3 décimales dans ta copie) ;
- d) vrai ou faux : la première ligne est $\boldsymbol{\mu} + \mathbf{L}\boldsymbol{\varepsilon}$, où $\boldsymbol{\varepsilon}$ est la première ligne de `np.random.default_rng(5921).standard_normal((20000, 3))`.

La vérification compare aussi `cov_21` à la covariance empirique des tirages attendus. Ce qu'il faut observer : les moments empiriques s'approchent de $\boldsymbol{\mu}$ et $\boldsymbol{\Sigma}$ à l'erreur type près (de l'ordre de $\frac{1}{\sqrt n}$), et le blanchiment rend un nuage de covariance proche de $\mathbf{I}$. Dans ta copie : l'écart entre `cov_21` et $\boldsymbol{\Sigma}$ est-il de l'ordre de l'erreur type d'une variance, $\Sigma_{jj}\sqrt{2/n}$ ?

In [ ]:
MU_ROAD_21 = np.array([40.0, 60.0, 30.0])                  # the road layers of M25a.2 (mm)
COV_ROAD_21 = np.array([[9.0, 3.0, -9.0], [3.0, 10.0, 0.0], [-9.0, 0.0, 11.0]])
COST_21 = np.array([2.0, 1.0, 3.0])                        # the cost c = 2 x1 + x2 + 3 x3 (€ per m²)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25a")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M25A, "test_sample_mvn_")

**L'usage** : 20 000 épaisseurs de route, leurs moments, leur coût, et leur blanchiment.

In [ ]:
sample_21 = cov_21 = answer_M25a_21a = answer_M25a_21b = answer_M25a_21c = answer_M25a_21d = ...
with wbm.attempt("M25a.21"):          # if you call sample_mvn here: ⏳ while it is not written
    sample_21 = ...        # your sample_mvn: 20 000 draws of the road layers, seed=5921 (an array (20000, 3))
    answer_M25a_21a = ...  # a) a list of 3 numbers, as computed: the empirical mean of sample_21
    cov_21 = ...           # a) its empirical covariance, np.cov(sample_21, rowvar=False, bias=True) (a 3 × 3 array)
    answer_M25a_21b = ...  # b) a list of 2 numbers, as computed: the empirical mean and variance (np.var) of the cost
    answer_M25a_21c = ...  # c) a 3 × 3 matrix, as computed: the empirical covariance (bias=True) of the whitened draws
    answer_M25a_21d = ...  # d) True or False: the first row is μ + L ε, ε the first row of default_rng(5921).standard_normal((20000, 3))

In [ ]:
with checked("M25a.21"):
    if filled(cov_21):
        E_c21 = np.random.default_rng(5921).standard_normal((20000, 3))
        draws_c21 = MU_ROAD_21 + E_c21 @ np.linalg.cholesky(COV_ROAD_21).T
        same("M25a.21", "cov_21, la covariance empirique de tes 20 000 tirages", cov_21,
             np.cov(draws_c21, rowvar=False, bias=True))
    else:
        print("⏳ Ex M25a.21 : la comparaison attend ta covariance cov_21.")
for letter, answer in zip("abcd", [answer_M25a_21a, answer_M25a_21b, answer_M25a_21c, answer_M25a_21d]):
    wbm.check(f"M25a.21{letter}", answer, computed=True)

### Ex M25a.22 — Le complément de Schur d'un bloc, et la loi conditionnelle d'un vecteur gaussien · `schur_complement`, `condition_gaussian` 🔨 ★★ ⏱️ 35 min
**Objectif :** programmer le complément de Schur et la loi conditionnelle d'un vecteur gaussien, puis vérifier leurs propriétés sur le capteur en panne.  
**Prérequis :** fiche §4 ; M09a ; Ex M25a.4 · **mymath :** `gaussian.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M25a.20 : écris la fonction dans ta copie `mon_travail/mymath/gaussian.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `schur_complement(M, idx)`, le complément de Schur du bloc `M[idx, idx]`, $\mathbf{A} - \mathbf{B}\mathbf{D}^{-1}\mathbf{C}$ avec les blocs de la docstring (fiche §4, 📐), par `np.linalg.solve` ; puis `condition_gaussian(mean, cov, idx, values)`, la loi conditionnelle des coordonnées hors de `idx` (dans l'ordre croissant) sachant `x[idx] = values` : le couple (moyennes conditionnelles, covariance conditionnelle). `ValueError` pour les indices et les blocs que refusent les docstrings.

**L'usage.** La cellule fournie donne le capteur en panne de M25a.4 (`MU_SENSOR_22`, `COV_SENSOR_22`, coordonnées 0, 1, 2), et les lectures `READ_22` des coordonnées `[0, 2]`. Avec tes fonctions :
- a) la loi de la coordonnée 1 sachant les coordonnées `[0, 2]` égales à 19,5 et 14 : la moyenne et la variance conditionnelles (une liste de 2 nombres, sans arrondir), à comparer à tes réponses c) et d) de M25a.4 ;
- b) vrai ou faux : `schur_complement(cov, [0, 2])` est la covariance conditionnelle de a) ;
- c) vrai ou faux : $\det\boldsymbol{\Sigma}$ (par `np.linalg.det`, hors de `mymath`) est égal à $\det\boldsymbol{\Sigma}_{bb}$ fois le déterminant du complément de Schur, à $10^{-12}$ près ;
- d) vrai ou faux : l'inverse du coefficient $(1, 1)$ de `np.linalg.inv(cov)` est égal à la variance conditionnelle ;
- e) les voisins lisent maintenant 21 °C et 12 °C (`READ_NEW_22`) : la nouvelle moyenne conditionnelle, et vrai ou faux, la variance conditionnelle est inchangée (une liste : le nombre, puis le booléen) ;
- f) vrai ou faux : `idx = [2, 0]` avec les valeurs dans l'ordre 14 et 19,5 donne la même loi qu'en a).

Ce qu'il faut observer : la moyenne conditionnelle suit les valeurs lues, la covariance conditionnelle n'en dépend pas, et c'est l'inverse du bloc de la précision. Dans ta copie : pourquoi les écarts $+3$ et $-3$ de e) déplacent-ils la moyenne de $+1$ ?

In [ ]:
MU_SENSOR_22 = np.array([18.0, 17.0, 15.0])                # the three sensors of M25a.4 (°C): coordinates 0, 1, 2
COV_SENSOR_22 = np.array([[4.0, 3.0, 1.0], [3.0, 4.0, 2.0], [1.0, 2.0, 4.0]])
READ_22 = [19.5, 14.0]                                     # the readings of the coordinates [0, 2]
READ_NEW_22 = [21.0, 12.0]                                 # e) other readings

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25a")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M25A, "test_schur_complement_ or test_condition_gaussian_")

**L'usage** : le capteur en panne, ses deux voisins, et d'autres lectures.

In [ ]:
answer_M25a_22a = answer_M25a_22b = answer_M25a_22c = answer_M25a_22d = answer_M25a_22e = answer_M25a_22f = ...
with wbm.attempt("M25a.22"):          # if you call your functions here: ⏳ while they are not written
    answer_M25a_22a = ...  # a) a list of 2 numbers, as computed: the conditional mean and variance of coordinate 1 given [0, 2] = READ_22
    answer_M25a_22b = ...  # b) True or False: schur_complement(COV_SENSOR_22, [0, 2]) is the conditional covariance of a)
    answer_M25a_22c = ...  # c) True or False: np.linalg.det(Σ) equals det Σ_bb × det(Schur complement), to 1e-12
    answer_M25a_22d = ...  # d) True or False: 1 / np.linalg.inv(Σ)[1, 1] equals the conditional variance of a)
    answer_M25a_22e = ...  # e) a list of 2: the new conditional mean for READ_NEW_22 (as computed), then True or False: the conditional variance is unchanged
    answer_M25a_22f = ...  # f) True or False: idx = [2, 0] with the values [14, 19.5] gives the same law as a)

In [ ]:
for letter, answer in zip("abcdef", [answer_M25a_22a, answer_M25a_22b, answer_M25a_22c, answer_M25a_22d,
                                        answer_M25a_22e, answer_M25a_22f]):
    wbm.check(f"M25a.22{letter}", answer, computed=True)

### Ex M25a.23 — Le posterior d'une régression linéaire bayésienne · `linreg_posterior` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la loi a posteriori d'une régression linéaire bayésienne, puis l'appliquer au thermomètre : données une à une, lien avec ridge, et prédictive près et loin des points d'étalonnage.  
**Prérequis :** fiche §5 ; M09b ; Ex M25a.6 · **mymath :** `gaussian.py` · **Parcours :** C

> Même mode d'emploi qu'en M25a.20 : écris la fonction dans ta copie `mon_travail/mymath/gaussian.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `linreg_posterior(Phi, y, prior_mean, prior_cov, noise_var)` (fiche §5), en suivant sa docstring : la loi a posteriori $\mathcal{N}(\mathbf{m}_N, \mathbf{S}_N)$ des poids, par la forme « précisions », $\mathbf{S}_N = (\mathbf{S}_0^{-1} + \sigma^{-2}\boldsymbol{\Phi}^\top\boldsymbol{\Phi})^{-1}$ et $\mathbf{m}_N = \mathbf{S}_N(\mathbf{S}_0^{-1}\mathbf{m}_0 + \sigma^{-2}\boldsymbol{\Phi}^\top\mathbf{y})$, avec `np.linalg.solve` (sans données, $n = 0$, elle rend l'a priori). Le couple `(m_N, S_N)` ; `ValueError` pour les formes, les covariances et les variances que refuse la docstring (`noise_var` est une **variance**).

**L'usage.** La cellule fournie donne le thermomètre de M25a.6 (`PHI_23`, `E_23`, `S0_23`, `NOISE_23`) et les températures `TS_23`, de −50 à 250 °C. Avec ta fonction :
- `posterior_23` : la loi a posteriori de $(a, b)$ ; a) $\mathbf{m}_N$ (une liste de 2 nombres), à comparer à ta réponse e) de M25a.6 ; la vérification compare aussi $\mathbf{S}_N$ à ta réponse d) ;
- b) vrai ou faux : en traitant les deux bains l'un après l'autre (la loi a posteriori après le premier sert d'a priori pour le second), on retrouve a) à $10^{-12}$ près, $\mathbf{m}_N$ et $\mathbf{S}_N$ ;
- c) avec l'a priori $\mathcal{N}(\mathbf{0}, 0{,}5\,\mathbf{I})$ : $\mathbf{m}_N$ (4 décimales dans ta copie), et vrai ou faux, il est égal à la solution ridge `np.linalg.solve(Phi.T @ Phi + lam * np.eye(2), Phi.T @ e)` de pénalité $\lambda = \frac{\sigma^2}{\tau^2}$ (une liste : les deux nombres, puis le booléen) ;
- d) avec l'a priori de M25a.6, la température $T$ (en °C) où la variance de la prédictive est la plus petite, par la formule $-100\,\frac{(\mathbf{S}_N)_{12}}{(\mathbf{S}_N)_{22}}$ (1 décimale dans ta copie ; $(\mathbf{S}_N)_{12}$ s'écrit `S_N[0, 1]` en Python) ;
- `pred_means_23` et `pred_sds_23` : la moyenne et l'écart-type de la prédictive de l'erreur d'une lecture (bruit compris) aux températures `TS_23`, puis `draw_predictive_23(ax, ts, means, sds)`, qui trace dans l'axe reçu la moyenne, sa bande à deux écarts-types (`ax.fill_between`) et les deux points d'étalonnage.

La vérification compare tes tableaux à la formule de la fiche, puis appelle `draw_predictive_23` et regarde le dessin. Ce qu'il faut observer : les données s'ajoutent une à une sans rien changer au résultat ; l'incertitude est la plus petite entre les bains et grandit vite en extrapolation. Dans ta copie : pourquoi la température de d) n'est-elle pas exactement au milieu des deux bains ?

In [ ]:
PHI_23 = np.array([[1.0, 0.0], [1.0, 1.0]])               # the thermometer of M25a.6: features (1, T/100), baths 0 and 100 °C
E_23 = np.array([0.3, -0.6])                               # the errors read in the two baths (°C)
S0_23 = np.diag([0.25, 1.0])                               # the prior N(0, S0) of θ = (a, b)
NOISE_23 = 0.04                                            # σ², the variance of the reading noise
TS_23 = np.linspace(-50.0, 250.0, 301)                     # the temperatures of the predictive (°C)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25a")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M25A, "test_linreg_posterior_")

**L'usage** : le thermomètre, bain par bain, avec un autre a priori, et sa prédictive.

In [ ]:
def draw_predictive_23(ax, ts, means, sds):
    """In ax: the predictive mean of the error against ts, its band mean ± 2 sds (ax.fill_between), and the two
    calibration points (0, 0.3) and (100, -0.6); labels and a legend. No other axes."""
    raise NotImplementedError("draw_predictive_23 n'est pas encore écrite (04_indices.md si tu bloques)")


posterior_23 = pred_means_23 = pred_sds_23 = ...
answer_M25a_23a = answer_M25a_23b = answer_M25a_23c = answer_M25a_23d = ...
with wbm.attempt("M25a.23"):          # if you call linreg_posterior here: ⏳ while it is not written
    posterior_23 = ...     # linreg_posterior for the thermometer: the pair (m_N, S_N)
    answer_M25a_23a = ...  # a) a list of 2 numbers, as computed: m_N
    answer_M25a_23b = ...  # b) True or False: one bath, then the other (the first posterior as the prior), gives a) to 1e-12
    answer_M25a_23c = ...  # c) a list of 3: m_N for the prior N(0, 0.5 I) (2 numbers, as computed), then True or False: it is the ridge solution of λ = σ²/τ²
    answer_M25a_23d = ...  # d) a number, as computed: −100 S_N[0, 1] / S_N[1, 1] (prior of M25a.6)
    pred_means_23 = ...    # the predictive means of the error at the temperatures TS_23 (prior of M25a.6): 301 numbers
    pred_sds_23 = ...      # their standard deviations, the noise σ² included: 301 numbers

In [ ]:
with checked("M25a.23"):
    if filled(posterior_23):
        m_check_23, S_check_23 = pair("M25a.23", "posterior_23", posterior_23, names=("m_N", "S_N"))
        same("M25a.23", "S_N, la covariance a posteriori", S_check_23,
             np.linalg.inv(np.linalg.inv(S0_23) + PHI_23.T @ PHI_23 / NOISE_23))
    else:
        print("⏳ Ex M25a.23 : la comparaison attend ta loi a posteriori posterior_23.")
for letter, answer in zip("abcd", [answer_M25a_23a, answer_M25a_23b, answer_M25a_23c, answer_M25a_23d]):
    wbm.check(f"M25a.23{letter}", answer, computed=True)

with checked("M25a.23"):
    results_23 = []
    report_23 = reporter(results_23)
    if not filled(pred_means_23, pred_sds_23):
        print("⏳ Ex M25a.23 : la figure attend tes tableaux pred_means_23 et pred_sds_23.")
        raise StopCheck
    S_c23 = np.linalg.inv(np.linalg.inv(S0_23) + PHI_23.T @ PHI_23 / NOISE_23)
    m_c23 = S_c23 @ (PHI_23.T @ E_23 / NOISE_23)
    features_c23 = np.column_stack([np.ones_like(TS_23), TS_23 / 100])
    means_c23 = features_c23 @ m_c23
    sds_c23 = np.sqrt(np.einsum("ij,jk,ik->i", features_c23, S_c23, features_c23) + NOISE_23)
    results_23.append(same("M25a.23", "pred_means_23, les moyennes de la prédictive", pred_means_23, means_c23))
    results_23.append(same("M25a.23", "pred_sds_23, les écarts-types de la prédictive (bruit compris)", pred_sds_23,
                           sds_c23))
    if not all(results_23):
        raise StopCheck
    fig_23, ax_23 = plt.subplots(figsize=(8, 4.2))
    try:
        opened_23 = drawn(draw_predictive_23, ax_23, TS_23, means_c23, sds_c23)
        report_23(not opened_23, "draw_predictive_23 dessine dans l'axe reçu",
                  "draw_predictive_23 doit dessiner dans l'axe reçu, sans ouvrir de figure")
        report_23(through(ax_23, TS_23, means_c23), "la moyenne de la prédictive est tracée contre la température",
                  "il manque la courbe des moyennes contre TS_23 (une valeur par température)")
        band_23 = any(type(c).__name__ != "PathCollection" for c in ax_23.collections) or (
            through(ax_23, TS_23, means_c23 + 2 * sds_c23) and through(ax_23, TS_23, means_c23 - 2 * sds_c23))
        report_23(band_23, "la bande à deux écarts-types est tracée",
                  "il manque la bande moyenne ± 2 écarts-types (ax.fill_between, ou deux courbes)")
        report_23(has_points(ax_23, [(0.0, 0.3), (100.0, -0.6)]), "les deux points d'étalonnage sont placés",
                  "il manque les deux points d'étalonnage (0 ; 0,3) et (100 ; −0,6)")
    except StopCheck:
        plt.show()
        raise
    except BaseException:
        plt.close(fig_23)
        raise
    plt.show()
    verdict("M25a.23", all(results_23), "la prédictive et sa bande sont dessinées.", "")

### Ex M25a.24 — `scipy.stats.multivariate_normal`, `np.cov` et `scipy.integrate` : tes log-densités, le maximum de vraisemblance, et la KL de M25a.8 par une intégrale double 📦 ★ ⏱️ 25 min
**Objectif :** comparer ta librairie à SciPy, retrouver le maximum de vraisemblance avec `np.cov`, et calculer une KL entre deux gaussiennes de trois façons.  
**Prérequis :** fiche §1, §6 et §7 ; Ex M25a.20 et M25a.8 · **Parcours :** C, E

La cellule fournie donne les épaisseurs de route de M25a.2, la serre de M25a.1, et les deux réglages de M25a.8 ($p_0$ : `MU0_24`, `S0_24` ; $p_1$ : `MU1_24`, `S1_24`). Avec ta `sample_mvn` et ta `mvn_logpdf` (M25a.20 et M25a.21), et SciPy :
- a) tire 1 000 vecteurs des épaisseurs de route (`seed=5924`) : vrai ou faux, ta `mvn_logpdf` et `scipy.stats.multivariate_normal(mean, cov).logpdf` diffèrent de moins de $10^{-10}$ sur les 1 000 lignes ;
- b) 50 tirages de la serre (`sample_mvn(mean, cov, 50, seed=5924)`, le tableau $\mathbf{X}$) : l'estimation du maximum de vraisemblance $\hat{\boldsymbol{\Sigma}} = \frac1n(\mathbf{X} - \bar{\mathbf{x}})^\top(\mathbf{X} - \bar{\mathbf{x}})$, calculée à la main en NumPy (3 décimales dans ta copie), et vrai ou faux, elle est égale à `np.cov(X, rowvar=False, bias=True)` (une liste de 5 : les 4 coefficients de $\hat{\boldsymbol{\Sigma}}$ ligne par ligne, puis le booléen) ;
- c) le rapport entre `np.cov(X, rowvar=False)` (sans `bias`) et $\hat{\boldsymbol{\Sigma}}$, le même pour tous les coefficients (4 décimales dans ta copie) ;
- d) la KL $\mathrm{KL}(p_0 \,\|\, p_1)$ de M25a.8 par `scipy.integrate.dblquad`, en intégrant $p_0\ln\frac{p_0}{p_1}$ sur le carré $[\mu_{0,1} - 25,\ \mu_{0,1} + 25] \times [\mu_{0,2} - 25,\ \mu_{0,2} + 25]$ (4 décimales dans ta copie), à comparer à ta réponse d) de M25a.8 ;
- e) la même KL estimée par la moyenne, sur 200 000 tirages de $p_0$ (`sample_mvn(MU0_24, S0_24, 200000, seed=5924)`), de $\ln p_0 - \ln p_1$, puis l'erreur type de cette moyenne, l'écart-type des 200 000 termes divisé par $\sqrt{200\,000}$ (une liste de 2 nombres ; 4 décimales chacune dans ta copie).

Ce qu'il faut observer : la formule fermée, l'intégrale double et la moyenne sur des tirages donnent la même KL, cette dernière à son erreur type près ; et, sans `bias=True`, `np.cov(X, rowvar=False)` divise par $n - 1$ (sans `rowvar=False`, il prendrait les lignes pour des variables). Dans ta copie : `dblquad` appelle ta fonction `f(y, x)` avec $y$ en premier ; pourquoi le carré suffit-il ?

In [ ]:
MU_ROAD_24 = np.array([40.0, 60.0, 30.0])                  # the road layers of M25a.2
COV_ROAD_24 = np.array([[9.0, 3.0, -9.0], [3.0, 10.0, 0.0], [-9.0, 0.0, 11.0]])
MU_GREEN_24 = np.array([21.0, 65.0])                       # the greenhouse of M25a.1
COV_GREEN_24 = np.array([[4.0, -6.0], [-6.0, 34.0]])
MU0_24, S0_24 = np.array([2.0, -1.0]), np.array([[4.0, 2.0], [2.0, 3.0]])     # the setting A of M25a.8: p0
MU1_24, S1_24 = np.array([3.0, -2.0]), np.array([[6.0, -1.0], [-1.0, 2.0]])   # the setting B: p1

In [ ]:
answer_M25a_24a = answer_M25a_24b = answer_M25a_24c = answer_M25a_24d = answer_M25a_24e = ...
with wbm.attempt("M25a.24"):          # your sample_mvn and mvn_logpdf: ⏳ while they are not written
    answer_M25a_24a = ...  # a) True or False: on 1 000 road draws (seed=5924), your mvn_logpdf and scipy's logpdf differ by less than 1e-10 on every row
    answer_M25a_24b = ...  # b) a list of 5: the 4 coefficients of Σ̂ (by hand, 50 greenhouse draws, seed=5924), row by row, as computed, then True or False: Σ̂ equals np.cov(X, rowvar=False, bias=True)
    answer_M25a_24c = ...  # c) a number, as computed: np.cov(X, rowvar=False) divided by Σ̂ (the same ratio for every coefficient)
    answer_M25a_24d = ...  # d) a number, as computed: KL(p0 ‖ p1) by scipy.integrate.dblquad on the square of half side 25 around MU0_24
    answer_M25a_24e = ...  # e) a list of 2 numbers, as computed: the mean of ln p0 − ln p1 over 200 000 draws of p0 (seed=5924), then its standard error

In [ ]:
for letter, answer in zip("abcde", [answer_M25a_24a, answer_M25a_24b, answer_M25a_24c, answer_M25a_24d,
                                       answer_M25a_24e]):
    wbm.check(f"M25a.24{letter}", answer, computed=True)

## Partie B · Prédire, mesurer, dessiner

Fiche §3 et §4. La masse d'un ellipsoïde quand la dimension grandit, prédite puis mesurée ; une loi conditionnelle approchée par simulation ; les ellipses, une tranche et une projection, dessinées.

### Ex M25a.25 — Prédire la part des tirages dans l'ellipsoïde de Mahalanobis de même niveau quand la dimension passe de 2 à 10, puis mesurer 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir, puis mesurer, la masse d'un ellipsoïde de Mahalanobis de niveau fixé quand la dimension augmente.  
**Prérequis :** fiche §3 ; Ex M25a.3 et M25a.21 · **Parcours :** C, E

On garde le niveau de l'ellipse à 90 % du plan, $c^2 = 2\ln 10$ (`LEVEL_25`), et l'on regarde, en dimension $d$, la part des tirages de $\mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma}_d)$ tels que $\Delta^2 \le c^2$, avec $(\boldsymbol{\Sigma}_d)_{ij} = 0{,}6^{\lvert i - j\rvert}$ (`ar_cov_25(d)`). **Avant d'exécuter quoi que ce soit**, prédis :
- a) en dimension 10, cette part est : (A) encore de l'ordre de 90 % ; (B) nettement plus petite ; (C) nettement plus grande (une lettre) ;
- b) la première dimension où elle passe sous 50 % : (A) 3 ; (B) entre 5 et 7 ; (C) au-delà de 10 (une lettre) ;
- c) vrai ou faux : pour une dimension donnée, cette part dépend de la matrice $\boldsymbol{\Sigma}_d$ choisie.

Puis, une fois tes prédictions écrites et vérifiées, mesure, pour $d = 2, \ldots, 10$, la part sur 100 000 tirages (`sample_mvn(np.zeros(d), ar_cov_25(d), 100000, seed=5925)`, $\Delta^2$ par `np.linalg.solve`), compare-la à `scipy.stats.chi2(d).cdf(LEVEL_25)`, et relève :
- d) la part mesurée en dimension 10 (3 décimales dans ta copie) ;
- e) la première dimension où la part mesurée passe sous 50 % (un entier).

La vérification de l'expérience ne s'exécute qu'une fois tes trois prédictions écrites : elle compare tes parts à celles des tirages attendus et affiche la loi du khi-deux à côté. Ce qu'il faut observer : $\Delta^2$ suit la loi du khi-deux à $d$ degrés quelle que soit $\boldsymbol{\Sigma}_d$ ; à niveau fixé, la masse de l'ellipsoïde ne dépend que de la dimension.

In [ ]:
LEVEL_25 = 2 * np.log(10)                                 # c² of the 90 % ellipse of the plane
DIMS_25 = list(range(2, 11))                               # d = 2, ..., 10


def ar_cov_25(d):
    """The covariance (Σ_d)_ij = 0.6^|i − j| in dimension d."""
    index = np.arange(d)
    return 0.6 ** np.abs(index[:, None] - index[None, :])

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M25a_25a = ...  # a) "A", "B" or "C", written BEFORE the experiment: the share in dimension 10
prediction_M25a_25b = ...  # b) "A", "B" or "C": the first dimension where the share falls below 50 %
prediction_M25a_25c = ...  # c) True or False: for a given dimension, the share depends on the matrix Σ_d chosen

In [ ]:
for letter, prediction in zip("abc", [prediction_M25a_25a, prediction_M25a_25b, prediction_M25a_25c]):
    wbm.check(f"M25a.25{letter}", prediction)

**L'expérience.** Écris-la une fois tes trois prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
shares_25 = answer_M25a_25d = answer_M25a_25e = ...
with wbm.attempt("M25a.25"):          # your sample_mvn: ⏳ while it is not written
    shares_25 = ...        # the 9 measured shares for d = 2, ..., 10: sample_mvn(np.zeros(d), ar_cov_25(d), 100000, seed=5925), Δ² by np.linalg.solve
    answer_M25a_25d = ...  # d) a number, as computed: the share measured in dimension 10
    answer_M25a_25e = ...  # e) an integer: the first dimension where the measured share is below 0.5

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M25a_25a, prediction_M25a_25b, prediction_M25a_25c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    with checked("M25a.25"):
        if not filled(shares_25):
            print("⏳ Ex M25a.25 : la vérification attend tes parts mesurées shares_25.")
            raise StopCheck
        want_c25 = []
        for d_c25 in DIMS_25:
            S_c25 = ar_cov_25(d_c25)
            X_c25 = np.random.default_rng(5925).standard_normal((100000, d_c25)) @ np.linalg.cholesky(S_c25).T
            m2_c25 = np.einsum("ij,ij->i", X_c25, np.linalg.solve(S_c25, X_c25.T).T)
            want_c25.append(float(np.mean(m2_c25 <= LEVEL_25)))
        if same("M25a.25", "shares_25, les parts mesurées pour d = 2, …, 10", shares_25, want_c25):
            print("   d    mesurée   khi-deux")
            for d_c25, share_c25 in zip(DIMS_25, want_c25):
                print(f"  {d_c25:2d}    {share_c25:.3f}     {stats.chi2(d_c25).cdf(LEVEL_25):.3f}".replace(".", ","))
    for letter, answer in zip("de", [answer_M25a_25d, answer_M25a_25e]):
        wbm.check(f"M25a.25{letter}", answer, computed=True)

**Après l'expérience**, dans ta copie : pourquoi la part ne dépend-elle pas de $\boldsymbol{\Sigma}_d$ (blanchis) ? Que vaut la moyenne de $\Delta^2$ en dimension $d$, et comment l'expliques-tu avec la couronne du §3 ?

### Ex M25a.26 — Conditionner par simulation : garder les tirages dont une mesure tombe près d'une valeur, et comparer à la loi conditionnelle exacte 🔬 ★★ ⏱️ 25 min
**Objectif :** approcher une loi conditionnelle en gardant les tirages proches de la valeur lue, et voir le compromis entre biais et nombre de tirages gardés.  
**Prérequis :** fiche §4 ; Ex M25a.1, M25a.21 et M25a.22 · **Parcours :** C

Pour la serre de M25a.1 (`MU_GREEN_26`, `COV_GREEN_26`), on cherche la loi de l'humidité $H$ sachant que la température vaut $T = 23$ °C (`T_READ_26`).
- a) Calcule-la exactement avec ta `condition_gaussian` : la moyenne et la variance (une liste de 2 nombres).

Puis tire 200 000 relevés (`sample_mvn(MU_GREEN_26, COV_GREEN_26, 200000, seed=5926)`) et, pour chaque demi-largeur $h$ de `HS_26` (2, 1, 0,5, 0,2 et 0,05 °C), garde ceux où $\lvert T - 23\rvert < h$ : `table_26` a une ligne par $h$, dans cet ordre, avec leur nombre, et la moyenne et la variance empiriques (`np.var`) de leur humidité. Écris `draw_window_26(ax_mean, ax_var, hs, means, variances, exact_mean, exact_var)`, qui trace dans les deux axes reçus (sans en ouvrir d'autre) la moyenne et la variance empiriques contre $h$ (abscisse logarithmique), avec les valeurs exactes (des droites horizontales). Relève :
- b) le nombre de relevés gardés pour $h = 0{,}05$ (un entier) ;
- c) la moyenne empirique de l'humidité pour $h = 2$ (2 décimales dans ta copie) ;
- d) vrai ou faux : pour $h = 2$, la variance empirique dépasse la variance exacte de plus de 1.

La vérification compare ton tableau à celui des tirages attendus, puis appelle `draw_window_26` et regarde le dessin. Ce qu'il faut conclure : une fenêtre large garde beaucoup de relevés mais biaise la moyenne et gonfle la variance (elle mélange des températures différentes) ; une fenêtre étroite est presque sans biais mais garde peu de relevés (leur nombre est à peu près proportionnel à $h$) ; la loi conditionnelle exacte évite ce compromis.

In [ ]:
MU_GREEN_26 = np.array([21.0, 65.0])                       # the greenhouse of M25a.1: (T, H)
COV_GREEN_26 = np.array([[4.0, -6.0], [-6.0, 34.0]])
T_READ_26 = 23.0                                           # the temperature read (°C)
HS_26 = [2.0, 1.0, 0.5, 0.2, 0.05]                         # the half widths of the window (°C)

In [ ]:
def draw_window_26(ax_mean, ax_var, hs, means, variances, exact_mean, exact_var):
    """In ax_mean: the empirical means of H against hs (points joined), the x scale logarithmic, and the exact mean (a
    horizontal line); in ax_var: the same for the variances. Labels and a legend in each. No other axes."""
    raise NotImplementedError("draw_window_26 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M25a_26a = table_26 = answer_M25a_26b = answer_M25a_26c = answer_M25a_26d = ...
with wbm.attempt("M25a.26"):          # your condition_gaussian and sample_mvn: ⏳ while they are not written
    answer_M25a_26a = ...  # a) a list of 2 numbers, as computed: the exact mean and variance of H given T = 23 (your condition_gaussian)
    table_26 = ...         # one row [count, mean of H, variance of H (np.var)] per h of HS_26, in this order: an array (5, 3)
    answer_M25a_26b = ...  # b) an integer: the number of readings kept for h = 0.05
    answer_M25a_26c = ...  # c) a number, as computed: the empirical mean of H for h = 2
    answer_M25a_26d = ...  # d) True or False: for h = 2, the empirical variance of H exceeds the exact variance by more than 1

In [ ]:
for letter, answer in zip("abcd", [answer_M25a_26a, answer_M25a_26b, answer_M25a_26c, answer_M25a_26d]):
    wbm.check(f"M25a.26{letter}", answer, computed=True)

with checked("M25a.26"):
    results_26 = []
    report_26 = reporter(results_26)
    if not filled(table_26):
        print("⏳ Ex M25a.26 : la vérification attend ton tableau table_26.")
        raise StopCheck
    draws_c26 = MU_GREEN_26 + np.random.default_rng(5926).standard_normal((200000, 2)) @ np.linalg.cholesky(COV_GREEN_26).T
    want_c26 = []
    for h_c26 in HS_26:
        kept_c26 = draws_c26[np.abs(draws_c26[:, 0] - T_READ_26) < h_c26, 1]
        want_c26.append([len(kept_c26), kept_c26.mean(), kept_c26.var()])
    want_c26 = np.array(want_c26)
    results_26.append(same("M25a.26", "table_26, une ligne [nombre, moyenne, variance] par demi-largeur", table_26,
                           want_c26))
    if not results_26[-1]:
        raise StopCheck
    gain_c26 = COV_GREEN_26[1, 0] / COV_GREEN_26[0, 0]
    exact_c26 = (MU_GREEN_26[1] + gain_c26 * (T_READ_26 - MU_GREEN_26[0]), COV_GREEN_26[1, 1] - gain_c26 * COV_GREEN_26[0, 1])
    fig_26, (ax_mean_26, ax_var_26) = plt.subplots(1, 2, figsize=(11, 4))
    try:
        opened_26 = drawn(draw_window_26, ax_mean_26, ax_var_26, HS_26, want_c26[:, 1], want_c26[:, 2], *exact_c26)
        report_26(not opened_26, "draw_window_26 dessine dans les axes reçus",
                  "draw_window_26 doit dessiner dans les deux axes reçus, sans ouvrir de figure")
        report_26(through(ax_mean_26, HS_26, want_c26[:, 1]) and through(ax_var_26, HS_26, want_c26[:, 2]),
                  "les moyennes et les variances empiriques sont tracées contre h",
                  "il manque une courbe : les moyennes (à gauche), puis les variances (à droite), contre h")
        report_26(ax_mean_26.get_xscale() == "log" and ax_var_26.get_xscale() == "log", "les abscisses sont logarithmiques",
                  "mets les abscisses en échelle logarithmique (ax.set_xscale(\"log\"))")
        report_26(horizontal(ax_mean_26, exact_c26[0]) and horizontal(ax_var_26, exact_c26[1]),
                  "les valeurs exactes sont tracées", "il manque une droite horizontale : la moyenne exacte (à gauche) "
                  "ou la variance exacte (à droite), par ax.axhline")
    except StopCheck:
        plt.show()
        raise
    except BaseException:
        plt.close(fig_26)
        raise
    plt.show()
    verdict("M25a.26", all(results_26), "ton tableau est juste, et dessiné avec les valeurs exactes.", "")

**Après la figure**, dans ta copie : pourquoi la moyenne empirique est-elle biaisée pour $h = 2$, et dans quel sens ? Combien de relevés faudrait-il tirer pour en garder autant avec $h = 0{,}05$ qu'avec $h = 2$ ?

### Ex M25a.27 — Les ellipses de niveau d'une gaussienne en dimension 2, une tranche (la loi conditionnelle) et une projection (la loi marginale), selon la corrélation 🎨 ★ ⏱️ 20 min
**Objectif :** voir, selon la corrélation, que la tranche d'une gaussienne du plan est une loi normale décalée et plus étroite, et sa projection une loi normale de même variance.  
**Prérequis :** fiche §3 et §4 ; Ex M25a.22 · **Parcours :** C, E

Écris `draw_gaussian_27(ax_joint, ax_cut, mean, cov, x1_value)`, qui dessine sur les **axes reçus** (sans ouvrir de figure) :
- dans `ax_joint`, les ellipses de $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ qui contiennent les masses de `LEVELS_27` (75 % et 90 %), chacune obtenue en appliquant $\boldsymbol{\varepsilon} \mapsto \boldsymbol{\mu} + \mathbf{L}\boldsymbol{\varepsilon}$ au cercle de rayon $c$ (fiche §3, 💻 ; au moins 50 points, de $0$ à $2\pi$), la droite verticale $x_1 = $ `x1_value` (`ax.axvline`) et la droite des moyennes conditionnelles de $x_2$ sachant $x_1$, avec la même échelle sur les deux axes (`ax_joint.set_aspect("equal")`) ;
- dans `ax_cut`, la densité de la loi marginale de $x_2$ et celle de la loi conditionnelle de $x_2$ sachant $x_1 = $ `x1_value` (par ta `condition_gaussian`, `mymath.gaussian.condition_gaussian`), chacune sur au moins 50 points, avec une légende.

La cellule fournie donne $\boldsymbol{\mu} = (0, 1)$ (`MU_27`), les écarts-types 1 et 2 (`SDS_27`), les corrélations $\rho = 0$, $0{,}5$ et $0{,}9$ (`RHOS_27`), la covariance correspondante (`cov_27(rho)`) et $x_1 = 1{,}5$ (`X1_27`). La vérification appelle ta fonction pour les trois corrélations, trois colonnes de deux panneaux, puis pour un cas que tu ne vois pas, et regarde chaque dessin. Ce qu'il faut observer : la tranche est une loi normale décalée de $\rho\frac{\sigma_2}{\sigma_1}(x_1 - \mu_1)$ et plus étroite (variance multipliée par $1 - \rho^2$), la projection garde la variance $\sigma_2^2$ ; pour $\rho = 0$, les deux courbes se confondent ; et la droite des moyennes conditionnelles passe par les points où les ellipses ont une tangente verticale.

In [ ]:
MU_27 = np.array([0.0, 1.0])
SDS_27 = (1.0, 2.0)                                        # the standard deviations of x1 and x2
RHOS_27 = (0.0, 0.5, 0.9)                                  # the three correlations
X1_27 = 1.5                                                # the slice x1 = 1.5
LEVELS_27 = (0.75, 0.9)                                    # the masses of the two ellipses


def cov_27(rho):
    """The covariance of standard deviations SDS_27 and correlation rho."""
    s1, s2 = SDS_27
    return np.array([[s1 ** 2, rho * s1 * s2], [rho * s1 * s2, s2 ** 2]])

In [ ]:
def draw_gaussian_27(ax_joint, ax_cut, mean, cov, x1_value):
    """In ax_joint: the ellipses of N(mean, cov) that contain the masses LEVELS_27, each the image μ + L(c cos t, c sin t)
    of the circle of radius c (at least 50 points, t from 0 to 2π), the vertical line x1 = x1_value (ax.axvline), the
    line of the conditional means of x2 given x1, equal scales (ax_joint.set_aspect("equal")); in ax_cut: the density
    of the marginal law of x2 and that of the conditional law of x2 given x1 = x1_value (your condition_gaussian), each
    on at least 50 points, with a legend. No other axes."""
    raise NotImplementedError("draw_gaussian_27 n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
def look_27(ax_joint, ax_cut, mean, cov, x1_value):
    """What is missing in a drawing of draw_gaussian_27: a list of French phrases, empty when all is there."""
    mean, cov = np.asarray(mean, dtype=float), np.asarray(cov, dtype=float)
    L, missing = np.linalg.cholesky(cov), []
    for level in LEVELS_27:
        c2, found = -2 * math.log(1 - level), False
        for line in ax_joint.lines:
            x, y = line_points(line)
            if x.size < 20:
                continue
            white = np.linalg.solve(L, np.vstack([x - mean[0], y - mean[1]])).T
            if np.allclose(np.sum(white ** 2, axis=1), c2, rtol=1e-6, atol=1e-9):
                angles = np.sort(np.arctan2(white[:, 1], white[:, 0]))
                found = found or np.diff(np.concatenate([angles, [angles[0] + 2 * np.pi]])).max() < 0.5
        if not found:
            missing.append(f"l'ellipse à {round(100 * level)} % (des points de Δ² = c², tout autour du centre)")
    if not any(line_points(line)[0].size >= 2 and np.allclose(line_points(line)[0], x1_value, rtol=0, atol=1e-9)
               for line in ax_joint.lines):
        missing.append(f"la droite verticale x₁ = {fr(x1_value, 2)} (ax.axvline)")
    slope = cov[0, 1] / cov[0, 0]
    if not any(np.ptp(line_points(line)[0]) > 0 and np.allclose(
            line_points(line)[1], mean[1] + slope * (line_points(line)[0] - mean[0]), rtol=0, atol=1e-9)
               for line in ax_joint.lines):
        missing.append("la droite des moyennes conditionnelles de x₂ sachant x₁")
    if ax_joint.get_aspect() not in ("equal", 1.0):
        missing.append("la même échelle sur les deux axes du panneau du haut (set_aspect(\"equal\"))")
    gain = cov[1, 0] / cov[0, 0]
    laws = {"la densité de la loi marginale de x₂": (mean[1], cov[1, 1]),
            "la densité de la loi conditionnelle de x₂ sachant x₁": (mean[1] + gain * (x1_value - mean[0]),
                                                                    cov[1, 1] - gain * cov[0, 1])}
    for name, (m, v) in laws.items():
        if not any(line_points(line)[0].size >= 20 and np.allclose(
                line_points(line)[1], stats.norm(m, math.sqrt(v)).pdf(line_points(line)[0]), rtol=1e-6, atol=1e-12)
                   for line in ax_cut.lines):
            missing.append(name + " (panneau du bas)")
    if ax_cut.get_legend() is None:
        missing.append("la légende du panneau du bas")
    return missing


with checked("M25a.27"):
    results_27 = []
    report_27 = reporter(results_27)
    fig_27, axes_27 = plt.subplots(2, 3, figsize=(15, 9.5))
    try:
        for column_27, rho_27 in enumerate(RHOS_27):
            name_27 = f"ρ = {rho_27:g}".replace(".", ",")
            opened_27 = drawn(draw_gaussian_27, axes_27[0, column_27], axes_27[1, column_27], MU_27, cov_27(rho_27), X1_27)
            report_27(not opened_27, f"{name_27} : draw_gaussian_27 dessine dans les axes reçus",
                      f"{name_27} : draw_gaussian_27 doit dessiner dans les deux axes reçus, sans ouvrir de figure")
            missing_27 = look_27(axes_27[0, column_27], axes_27[1, column_27], MU_27, cov_27(rho_27), X1_27)
            report_27(not missing_27, f"{name_27} : les deux ellipses, la verticale, la droite des moyennes "
                      "conditionnelles et les deux densités sont là", f"{name_27} : il manque " + " ; ".join(missing_27))
        rng_c27 = np.random.default_rng(5917)
        sds_c27, rho_c27 = rng_c27.uniform(0.6, 2.5, size=2), -0.7
        cov_c27 = np.array([[sds_c27[0] ** 2, rho_c27 * sds_c27[0] * sds_c27[1]],
                            [rho_c27 * sds_c27[0] * sds_c27[1], sds_c27[1] ** 2]])
        mean_c27 = rng_c27.uniform(-3, 3, size=2)
        fig_c27, (joint_c27, cut_c27) = plt.subplots(1, 2)
        try:
            opened_c27 = drawn(draw_gaussian_27, joint_c27, cut_c27, mean_c27.copy(), cov_c27.copy(),
                               float(mean_c27[0] - 0.8 * sds_c27[0]))
            missing_c27 = look_27(joint_c27, cut_c27, mean_c27, cov_c27, float(mean_c27[0] - 0.8 * sds_c27[0]))
        finally:
            plt.close(fig_c27)
        report_27(not opened_c27 and not missing_c27, "un cas que tu ne vois pas (une autre moyenne, une corrélation "
                  "négative, une autre tranche) est dessiné juste aussi", "un cas que tu ne vois pas (une autre moyenne, "
                  "une corrélation négative, une autre tranche) : il manque " + " ; ".join(missing_c27 or ["…"]))
    except StopCheck:
        fig_27.tight_layout()
        plt.show()
        raise
    except BaseException:
        plt.close(fig_27)
        raise
    fig_27.tight_layout()
    plt.show()
    verdict("M25a.27", all(results_27), "les ellipses, les tranches et les projections sont dessinées.", "")

**Après la figure**, dans ta copie : où la droite des moyennes conditionnelles coupe-t-elle les ellipses ? Que deviennent la tranche et la projection quand $\rho$ tend vers 1 ? Quelle part de la variance de $x_2$ la lecture de $x_1$ enlève-t-elle, pour chaque $\rho$ ?

## Partie C · Déboguer, tester

Fiche §1, §2 et §4. Trois erreurs classiques qui ne lèvent aucune erreur Python ; un test statistique pour un tirage. La vérification de M25a.29 s'appuie sur la référence (`ref_gauss`), pour ne pas dépendre de ton code.

In [ ]:
ref_gauss = wbm.load_mymath("ref", alias="mymath_ref").gaussian   # the reference: the check of M25a.29

### Ex M25a.28 — Chasse à l'erreur : un tirage par E L au lieu de E Lᵀ, ln det Σ sans le facteur 2, et une covariance conditionnelle qui dépend de la valeur observée 🐛 ★ ⏱️ 20 min
**Objectif :** reconnaître trois erreurs classiques qui ne lèvent aucune erreur Python et passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §1, §2 et §4 ; Ex M25a.1, M25a.3 et M25a.4 · **Parcours :** C, E

Un collègue a écrit trois fonctions (cellule fournie) ; chacune a l'erreur que nomme le titre, aucune ne lève d'erreur Python, et chacune a passé la vérification qu'il a faite :
1. `draw_correlated(mean, cov, n, seed)` doit rendre $n$ tirages de $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ rangés en lignes ; elle calcule `L = np.linalg.cholesky(cov)` et `E = np.random.default_rng(seed).standard_normal((n, d))`, puis rend `mean + E @ L`. Il l'a vérifiée sur la loi de moyenne nulle et de covariance $\operatorname{diag}(4, 9)$ : la covariance empirique de 50 000 tirages est bien proche de $\boldsymbol{\Sigma}$. Il l'applique au vent de M25a.3 (`MU_WIND_28`, `COV_WIND_28`).
2. `log_density(x, mean, cov)` doit rendre la log-densité de $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ en un point ; elle calcule $\mathbf{L}$, résout $\mathbf{L}\mathbf{y} = \mathbf{x} - \boldsymbol{\mu}$, puis rend $-\frac12\big[d\ln(2\pi) + \sum_i\ln L_{ii} + \lVert\mathbf{y}\rVert^2\big]$. Il l'a vérifiée en trois points d'une loi de dimension 2 : `log_density(x, mean, cov) - log_density(mean, mean, cov)` vaut bien $-\frac12\Delta^2(\mathbf{x})$. Il l'applique à la serre de M25a.1 (`MU_GREEN_28`, `COV_GREEN_28`, `XP_28`).
3. `conditional_law(mean, cov, idx, values)` doit rendre la moyenne et la covariance des coordonnées hors de `idx` sachant `x[idx] = values` ; pour « tenir compte de l'écart observé », elle remplace $\boldsymbol{\Sigma}_{bb}$ par la dispersion de $\mathbf{x}_b$ autour de la valeur lue, $\boldsymbol{\Sigma}_{bb} + (\mathbf{v} - \boldsymbol{\mu}_b)(\mathbf{v} - \boldsymbol{\mu}_b)^\top$ (avec $\mathbf{v}$ = `values`), dans la moyenne comme dans la covariance. Il l'a vérifiée sur le capteur en panne de M25a.4 (`MU_SENSOR_28`, `COV_SENSOR_28`), en supposant que les deux voisins lisaient leur moyenne. Il l'applique aux vraies lectures, 19,5 °C et 14 °C.

À faire : écris `draw_correlated_fixed`, `log_density_fixed` et `conditional_law_fixed`, avec les mêmes arguments (la troisième reçoit `idx` comme une liste, et ne modifie jamais les tableaux reçus), puis réponds :
- a) la covariance théorique des tirages que rend la version fausse de `draw_correlated` pour le vent (une matrice $2 \times 2$ ; 4 décimales dans ta copie) ;
- b) vrai ou faux : la covariance empirique (`bias=True`) de 50 000 tirages de la version corrigée (`seed=5928`) est à moins de 0,3 de $\boldsymbol{\Sigma}$, coefficient par coefficient ;
- c) la valeur que rend la version fausse de `log_density` en $\mathbf{x}_P$ (4 décimales dans ta copie) ;
- d) celle de la version corrigée (4 décimales dans ta copie) ;
- e) la moyenne et la variance conditionnelles que rend la version fausse de `conditional_law` (une liste de 2 nombres ; 4 décimales dans ta copie) ;
- f) celles de la version corrigée (une liste de 2 nombres ; 4 décimales dans ta copie).

Pour chacune des trois erreurs, dis aussi, dans ta copie, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie tes corrections sur des cas que tu ne vois pas : des lois de covariance pleine en dimension 1 à 5, des points quelconques, et des valeurs lues loin de la moyenne.

In [ ]:
MU_WIND_28 = np.array([2.5, 1.0])                          # the wind of M25a.3 (m/s)
COV_WIND_28 = np.array([[14.0, 3.0], [3.0, 6.0]])
MU_GREEN_28 = np.array([21.0, 65.0])                       # the greenhouse of M25a.1
COV_GREEN_28 = np.array([[4.0, -6.0], [-6.0, 34.0]])
XP_28 = np.array([23.0, 68.0])
MU_SENSOR_28 = np.array([18.0, 17.0, 15.0])                # the sensors of M25a.4
COV_SENSOR_28 = np.array([[4.0, 3.0, 1.0], [3.0, 4.0, 2.0], [1.0, 2.0, 4.0]])


# The colleague's code
def draw_correlated(mean, cov, n, seed):
    """n draws of N(mean, cov), one per row."""
    mean, cov = np.asarray(mean, dtype=float), np.asarray(cov, dtype=float)
    L = np.linalg.cholesky(cov)
    E = np.random.default_rng(seed).standard_normal((n, len(mean)))
    return mean + E @ L


def log_density(x, mean, cov):
    """ln N(x | mean, cov) at one point x."""
    x, mean, cov = (np.asarray(v, dtype=float) for v in (x, mean, cov))
    L = np.linalg.cholesky(cov)
    y = np.linalg.solve(L, x - mean)
    return float(-0.5 * (len(x) * np.log(2 * np.pi) + np.sum(np.log(np.diag(L))) + y @ y))


def conditional_law(mean, cov, idx, values):
    """(mean, covariance) of the coordinates not in idx, given x[idx] = values."""
    mean, cov, values = (np.asarray(v, dtype=float) for v in (mean, cov, values))
    rest = [i for i in range(len(mean)) if i not in idx]
    gap = values - mean[idx]
    S_bb = cov[np.ix_(idx, idx)] + np.outer(gap, gap)          # "to take the observed gap into account"
    S_ab = cov[np.ix_(rest, idx)]
    gain = S_ab @ np.linalg.inv(S_bb)
    return mean[rest] + gain @ gap, cov[np.ix_(rest, rest)] - gain @ S_ab.T


# The colleague's checks: all pass
check_1_28 = draw_correlated(np.zeros(2), np.diag([4.0, 9.0]), 50000, 5911)
print("(1) N(0, diag(4, 9)), 50 000 tirages : covariance empirique",
      np.round(np.cov(check_1_28, rowvar=False, bias=True), 2).tolist(), "(attendu proche de [[4, 0], [0, 9]])")
mean_2_28, cov_2_28 = np.array([-1.0, 0.5]), np.array([[2.0, -0.8], [-0.8, 1.25]])
ok_2_28 = all(np.isclose(log_density(x, mean_2_28, cov_2_28) - log_density(mean_2_28, mean_2_28, cov_2_28),
                          -0.5 * (x - mean_2_28) @ np.linalg.solve(cov_2_28, x - mean_2_28))
              for x in (np.array([0.0, 0.0]), np.array([1.5, -1.0]), np.array([-2.0, 2.0])))
print("(2) en trois points, log_density(x) − log_density(μ) = −½ Δ²(x) :", ok_2_28, "(attendu True)")
mean_3_28, cov_3_28 = conditional_law(MU_SENSOR_28, COV_SENSOR_28, [0, 2], MU_SENSOR_28[[0, 2]])
print("(3) voisins lus à leur moyenne : moyenne conditionnelle égale à μ₂ :", bool(np.isclose(mean_3_28[0], 17.0)),
      "; variance conditionnelle plus petite que Σ₂₂ :", bool(cov_3_28[0, 0] < COV_SENSOR_28[1, 1]), "(attendu True, True)")

In [ ]:
def draw_correlated_fixed(mean, cov, n, seed):
    """n draws of N(mean, cov), one per row: L = np.linalg.cholesky(cov), E = np.random.default_rng(seed).standard_normal((n, d))."""
    raise NotImplementedError("draw_correlated_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def log_density_fixed(x, mean, cov):
    """ln N(x | mean, cov) at one point x: a float."""
    raise NotImplementedError("log_density_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def conditional_law_fixed(mean, cov, idx, values):
    """(mean, covariance) of the coordinates not in idx (in increasing order), given x[idx] = values (idx a list)."""
    raise NotImplementedError("conditional_law_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M25a_28a = answer_M25a_28b = answer_M25a_28c = answer_M25a_28d = answer_M25a_28e = answer_M25a_28f = ...
with wbm.attempt("M25a.28"):          # your three functions: ⏳ while they are not written
    answer_M25a_28a = ...  # a) a 2 × 2 matrix, as computed: the theoretical covariance of the draws of the WRONG draw_correlated for the wind
    answer_M25a_28b = ...  # b) True or False: the empirical covariance (bias=True) of 50 000 draws of draw_correlated_fixed (seed=5928) is within 0.3 of Σ, coefficient by coefficient
    answer_M25a_28c = ...  # c) a number, as computed: the WRONG log_density at XP_28 (the greenhouse)
    answer_M25a_28d = ...  # d) a number, as computed: log_density_fixed at XP_28
    answer_M25a_28e = ...  # e) a list of 2 numbers, as computed: the conditional mean and variance from the WRONG conditional_law (readings 19.5 and 14)
    answer_M25a_28f = ...  # f) a list of 2 numbers, as computed: the same from conditional_law_fixed

In [ ]:
for letter, answer in zip("abcdef", [answer_M25a_28a, answer_M25a_28b, answer_M25a_28c, answer_M25a_28d,
                                        answer_M25a_28e, answer_M25a_28f]):
    wbm.check(f"M25a.28{letter}", answer, computed=True)

results_28, parts_28 = [], []


def report_28(ok, success, failure):
    results_28.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_28(part, name, function, *args):
    """function(*args); an error of the learner's code becomes one ❌ line that names the function (the check of this
    part stops there, the final verdict stays)."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", RuntimeWarning)
            return function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_28(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        parts_28.append(part)
        raise StopCheck from None


rng_c28 = np.random.default_rng(5918)


def random_law_28(d):
    """A random mean and a random full covariance (symmetric positive definite) in dimension d."""
    A = rng_c28.normal(size=(d, d))
    return rng_c28.normal(size=d) * 3, A @ A.T + 0.5 * np.eye(d)


with checked("M25a.28"):                                      # (1) draw_correlated_fixed
    wrong_28 = []
    for d_c28, n_c28, seed_c28 in ((2, 6, 5932), (3, 5, 5933), (4, 4, 5934)):
        mean_c28, cov_c28 = random_law_28(d_c28)
        got_c28 = call_28(1, "draw_correlated_fixed", draw_correlated_fixed, mean_c28.copy(), cov_c28.copy(), n_c28,
                          seed_c28)
        want_c28 = (mean_c28 + np.random.default_rng(seed_c28).standard_normal((n_c28, d_c28))
                    @ np.linalg.cholesky(cov_c28).T)
        if got_c28 is None or np.shape(got_c28) != want_c28.shape or not np.allclose(got_c28, want_c28, rtol=1e-9,
                                                                                    atol=1e-12):
            wrong_28.append(f"une loi de dimension {d_c28}, {n_c28} tirages : attendu un tableau {want_c28.shape} qui "
                            f"commence par {np.round(want_c28[0], 4).tolist()}, reçu {shown(got_c28)[:90]}")
    report_28(not wrong_28, "(1) draw_correlated_fixed : trois lois de covariance pleine, en dimension 2, 3 et 4 "
              "(comparées à μ + L εᵢ, même graine)", "(1) draw_correlated_fixed, " + " ; ".join(wrong_28[:2]))
    parts_28.append(1)

with checked("M25a.28"):                                      # (2) log_density_fixed
    wrong_28 = []
    for d_c28 in (1, 2, 3, 5):
        mean_c28, cov_c28 = random_law_28(d_c28)
        x_c28 = mean_c28 + rng_c28.normal(size=d_c28) * 2
        got_c28 = call_28(2, "log_density_fixed", log_density_fixed, x_c28.copy(), mean_c28.copy(), cov_c28.copy())
        want_c28 = float(stats.multivariate_normal(mean_c28, cov_c28).logpdf(x_c28))
        if got_c28 is None or np.ndim(got_c28) != 0 or not math.isclose(float(got_c28), want_c28, rel_tol=1e-9):
            wrong_28.append(f"une loi de dimension {d_c28} : attendu {want_c28:.6f}, reçu {shown(got_c28, 6)}"[:200])
    report_28(not wrong_28, "(2) log_density_fixed : des lois de dimension 1, 2, 3 et 5, en des points quelconques",
              "(2) log_density_fixed, " + " ; ".join(wrong_28[:2]))
    parts_28.append(2)

with checked("M25a.28"):                                      # (3) conditional_law_fixed
    wrong_28 = []
    for d_c28, idx_c28 in ((3, [1]), (4, [3, 0]), (5, [0, 2, 4])):
        mean_c28, cov_c28 = random_law_28(d_c28)
        values_c28 = mean_c28[idx_c28] + rng_c28.normal(size=len(idx_c28)) * 4      # far from the mean
        rest_c28 = [i for i in range(d_c28) if i not in idx_c28]
        precision_c28 = np.linalg.inv(cov_c28)
        cov_want_c28 = np.linalg.inv(precision_c28[np.ix_(rest_c28, rest_c28)])          # Σ_{a|b} = Λ_aa⁻¹
        mean_want_c28 = mean_c28[rest_c28] - cov_want_c28 @ precision_c28[np.ix_(rest_c28, idx_c28)] @ (
            values_c28 - mean_c28[idx_c28])
        got_c28 = call_28(3, "conditional_law_fixed", conditional_law_fixed, mean_c28.copy(), cov_c28.copy(),
                          list(idx_c28), values_c28.copy())
        if not isinstance(got_c28, (tuple, list)) or len(got_c28) != 2:
            wrong_28.append(f"attendu le couple (moyenne, covariance), reçu {shown(got_c28)[:90]}")
            break
        if (np.shape(got_c28[0]) != mean_want_c28.shape or np.shape(got_c28[1]) != cov_want_c28.shape
                or not np.allclose(got_c28[0], mean_want_c28, rtol=1e-9, atol=1e-12)
                or not np.allclose(got_c28[1], cov_want_c28, rtol=1e-9, atol=1e-12)):
            wrong_28.append(f"dimension {d_c28}, idx = {idx_c28}, des valeurs loin de la moyenne : attendu "
                            f"({np.round(mean_want_c28, 4).tolist()}, {np.round(cov_want_c28, 4).tolist()}), reçu "
                            f"{shown(tuple(got_c28))}"[:260])
    report_28(not wrong_28, "(3) conditional_law_fixed : des lois de dimension 3, 4 et 5, des valeurs lues loin de la "
              "moyenne", "(3) conditional_law_fixed, " + " ; ".join(wrong_28[:2]))
    parts_28.append(3)

if len(set(parts_28)) == 3:
    verdict("M25a.28", all(results_28), "les trois fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque erreur, pourquoi la vérification du collègue ne la voyait-elle pas (une covariance diagonale, des différences de log-densités, des voisins lus à leur moyenne), et quelle vérification l'aurait vue ? Pour (2), de combien la version fausse se trompe-t-elle, et pourquoi cet écart ne dépend-il pas du point ? Pour (3), dans quel sens la version fausse déplace-t-elle la moyenne, et que devient son erreur quand les lectures s'éloignent des moyennes ?

### Ex M25a.29 — Un test de propriété : la moyenne et la covariance empiriques de tes tirages, à une tolérance fixée par l'erreur type 🛠️ ★ ⏱️ 20 min
**Objectif :** écrire un test statistique dont la tolérance est fixée par l'erreur type et un risque d'échec annoncé, puis mesurer ce risque et la puissance du test.  
**Prérequis :** fiche §2 ; M02c ; Ex M25a.21 · **Parcours :** C

Écris `moment_test_29(sampler, mean, cov, n, seed, k)`, qui tire `X = sampler(mean, cov, n, seed)` et vérifie, par des `assert`, que chaque moyenne empirique est à moins de $k$ erreurs types de $\mu_j$, l'erreur type valant $\sqrt{\Sigma_{jj}/n}$, et que chaque coefficient de la covariance empirique (`bias=True`) est à moins de $k$ erreurs types de $\Sigma_{jk}$, l'erreur type valant $\sqrt{(\Sigma_{jj}\Sigma_{kk} + \Sigma_{jk}^2)/n}$ ; un échec lève `AssertionError`, avec un message qui dit **d'abord** ce qui était attendu (il commence par « expected »), **puis** le coefficient fautif. On prend la loi du vent de M25a.3 (`MU_WIND_29`, `COV_WIND_29`) et $n = 2\,000$ (`N_29`).
- a) Choisis $k$ pour que la probabilité qu'une fonction juste échoue soit au plus 1 % : en dimension 2, le test fait 5 comparaisons (2 moyennes et 3 coefficients de covariance) ; partage les 1 % entre elles (une borne de la réunion) et lis $k$ dans `scipy.stats.norm.ppf` (4 décimales dans ta copie).
- b) Lance ton test sur ta `sample_mvn` pour les 300 graines `SEEDS_29` (tirées par `np.random.default_rng(5929).integers(0, 2**31, size=300)`) : le nombre d'échecs, à comparer à ce que garantit a) (un entier).
- c) Sur ces 300 répétitions, mesure l'écart-type (`np.std`) du coefficient $(0, 1)$ de la covariance empirique, et compare-le à la formule de l'erreur type (une liste de 2 nombres ; 4 décimales chacun dans ta copie).
- d) La cellule fournie donne quatre versions fausses : `sampler_EL` (qui rend `mean + E @ L`), `sampler_sigma` (`mean + E @ cov`), `sampler_diag` (qui ignore les covariances : chaque coordonnée a la bonne variance, sans corrélation) et `sampler_inflated` (qui tire $\mathcal{N}(\boldsymbol{\mu}, 1{,}02\,\boldsymbol{\Sigma})$) ; lance ton test sur chacune avec la graine 5929 et $n = 2\,000$ : lesquelles attrape-t-il ? (une liste de 4 booléens, dans cet ordre : `True` quand ton test l'attrape).
- e) Combien de tirages faudrait-il pour qu'un écart de 2 % sur une variance dépasse $k$ erreurs types (l'erreur type d'une variance empirique vaut $\Sigma_{jj}\sqrt{2/n}$) ? Le plus petit entier $n$ (avec $k$ non arrondi). Vérifie que ton test attrape `sampler_inflated` avec $n = 300\,000$.

La vérification lance ton test sur la référence (`ref_gauss`), où il doit passer, sur les versions fausses, où il doit lever une `AssertionError` qui sort de ta fonction, sur un cas en dimension 3 que tu ne vois pas, puis sur **ta** librairie. Ce qu'il faut retenir : la tolérance d'un test statistique se choisit par l'erreur type et un risque d'échec annoncé, pas au hasard ; un tel test attrape les erreurs grossières à petit $n$, et une erreur fine seulement si $n$ est assez grand pour elle.

In [ ]:
MU_WIND_29 = np.array([2.5, 1.0])                          # the wind of M25a.3 (m/s)
COV_WIND_29 = np.array([[14.0, 3.0], [3.0, 6.0]])
N_29 = 2000
SEEDS_29 = np.random.default_rng(5929).integers(0, 2**31, size=300)   # the 300 seeds of b) and c)


def sampler_EL(mean, cov, n, seed):
    """Wrong: mean + E @ L."""
    E = np.random.default_rng(seed).standard_normal((n, len(mean)))
    return np.asarray(mean, dtype=float) + E @ np.linalg.cholesky(cov)


def sampler_sigma(mean, cov, n, seed):
    """Wrong: mean + E @ cov."""
    E = np.random.default_rng(seed).standard_normal((n, len(mean)))
    return np.asarray(mean, dtype=float) + E @ np.asarray(cov, dtype=float)


def sampler_diag(mean, cov, n, seed):
    """Wrong: each coordinate has the right variance, without any correlation."""
    E = np.random.default_rng(seed).standard_normal((n, len(mean)))
    return np.asarray(mean, dtype=float) + E * np.sqrt(np.diag(cov))


def sampler_inflated(mean, cov, n, seed):
    """Wrong by a little: draws of N(mean, 1.02 cov)."""
    E = np.random.default_rng(seed).standard_normal((n, len(mean)))
    return np.asarray(mean, dtype=float) + E @ np.linalg.cholesky(1.02 * np.asarray(cov, dtype=float)).T

In [ ]:
def moment_test_29(sampler, mean, cov, n, seed, k):
    """X = sampler(mean, cov, n, seed); assert that each empirical mean is within k standard errors sqrt(Σ_jj / n) of
    μ_j, and that each coefficient of the empirical covariance (bias=True) is within k standard errors
    sqrt((Σ_jj Σ_kk + Σ_jk²) / n) of Σ_jk. Each message says first what was expected ("expected ..."), then the
    faulty coefficient."""
    raise NotImplementedError("moment_test_29 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M25a_29a = answer_M25a_29b = answer_M25a_29c = answer_M25a_29d = answer_M25a_29e = ...
with wbm.attempt("M25a.29"):          # your sample_mvn and your test: ⏳ while they are not written
    answer_M25a_29a = ...  # a) a number, as computed: k (scipy.stats.norm.ppf)
    answer_M25a_29b = ...  # b) an integer: the number of failures (AssertionError) of your test on your sample_mvn, the wind, n = N_29, the 300 SEEDS_29
    answer_M25a_29c = ...  # c) a list of 2 numbers, as computed: the standard deviation (np.std) of the coefficient (0, 1) of the empirical covariance over the 300 draws, then the formula
    answer_M25a_29d = ...  # d) a list of 4 True or False, in the order sampler_EL, sampler_sigma, sampler_diag, sampler_inflated: True when your test catches it (seed 5929, n = N_29)
    answer_M25a_29e = ...  # e) an integer: the smallest n for which a gap of 2 % on a variance exceeds k standard errors

In [ ]:
for letter, answer in zip("abcde", [answer_M25a_29a, answer_M25a_29b, answer_M25a_29c, answer_M25a_29d,
                                       answer_M25a_29e]):
    wbm.check(f"M25a.29{letter}", answer, computed=True)

with checked("M25a.29"):
    results_29 = []
    report_29 = reporter(results_29, stop=False)
    K_c29 = float(stats.norm.ppf(1 - 0.01 / (2 * 5)))

    def run_29(sampler, label, mean=MU_WIND_29, cov=COV_WIND_29, n=N_29, seed=5929):
        """None when moment_test_29 passes, else the message of its AssertionError (other errors: ❌, then stop)."""
        try:
            moment_test_29(sampler, mean, cov, n, seed, K_c29)
        except AssertionError as exc:
            return str(exc) or "(un assert sans message)"
        except (NotImplementedError, MymathMissingError):
            raise
        except Exception as exc:  # noqa: BLE001 - another error than an assert
            print(f"❌ Ex M25a.29 : moment_test_29({label}) lève {type(exc).__name__} au lieu d'un assert : "
                  f"{str(exc)[:200]}")
            raise StopCheck from None
        return None

    message_29 = run_29(ref_gauss.sample_mvn, "référence")
    report_29(message_29 is None, "sur la référence (le vent, n = 2 000, graine 5929, k de a)), ton test passe",
              f"ton test échoue sur la référence, qui est juste : {message_29!r:.200}")
    for sampler_c29, label_c29 in ((sampler_EL, "sampler_EL"), (sampler_sigma, "sampler_sigma"),
                                   (sampler_diag, "sampler_diag")):
        message_29 = run_29(sampler_c29, label_c29)
        if message_29 is None:
            report_29(False, "", f"ton test n'attrape pas {label_c29} : il doit lever une AssertionError, et la laisser "
                      "sortir de ta fonction")
            continue
        report_29(message_29.strip().lower().startswith(("expected", "attendu")),
                  f"ton test attrape {label_c29} : {message_29[:140]}",
                  f"ton test attrape {label_c29}, mais son message doit commencer par ce qui était attendu "
                  f"(« expected … »), reçu : {message_29[:140]!r}")
    message_29 = run_29(sampler_inflated, "sampler_inflated")
    report_29(message_29 is None, "sampler_inflated (1,02 Σ) passe à n = 2 000, comme annoncé",
              f"ton test attrape déjà sampler_inflated à n = 2 000, ce que k de a) ne permet pas : relis tes erreurs "
              f"types ({message_29!r:.140})")
    message_29 = run_29(sampler_inflated, "sampler_inflated, n = 300 000", n=300000)
    report_29(message_29 is not None, "à n = 300 000, ton test attrape sampler_inflated",
              "à n = 300 000, ton test laisse passer sampler_inflated : vérifie l'erreur type des variances")
    rng_c29 = np.random.default_rng(5919)
    A_c29 = rng_c29.normal(size=(3, 3))
    cov3_c29, mean3_c29 = A_c29 @ A_c29.T + np.eye(3), rng_c29.normal(size=3) * 2
    message_29 = run_29(ref_gauss.sample_mvn, "référence, dimension 3", mean3_c29, cov3_c29, 3000, 5931)
    report_29(message_29 is None, "en dimension 3 (un cas que tu ne vois pas), ton test passe sur la référence",
              f"en dimension 3 (un cas que tu ne vois pas), ton test échoue sur la référence : {message_29!r:.200}")

    def shifted_c29(mean, cov, n, seed):
        """Wrong: the right draws, the second mean moved by 8 standard errors."""
        return ref_gauss.sample_mvn(mean, cov, n, seed) + np.array([0.0, 8 * math.sqrt(cov[1, 1] / n), 0.0])

    def flipped_c29(mean, cov, n, seed):
        """Wrong: the covariances of the first coordinate with the others of the wrong sign."""
        D = np.diag([-1.0, 1.0, 1.0])
        return ref_gauss.sample_mvn(mean, D @ cov @ D, n, seed) * np.array([1.0, 1.0, 1.0])

    for sampler_c29, label_c29 in ((shifted_c29, "une moyenne décalée"), (flipped_c29, "des covariances de mauvais signe")):
        message_29 = run_29(sampler_c29, label_c29, mean3_c29, cov3_c29, 3000, 5931)
        report_29(message_29 is not None, f"en dimension 3, ton test attrape une version fausse que tu ne vois pas "
                  f"({label_c29})", f"en dimension 3, ton test laisse passer une version fausse que tu ne vois pas "
                  f"({label_c29}) : compare toutes les moyennes et tous les coefficients de la covariance")
    try:
        moment_test_29(mymath.gaussian.sample_mvn, MU_WIND_29, COV_WIND_29, N_29, 5929, K_c29)
        print("✅ ta librairie (sample_mvn) passe ton test")
    except (NotImplementedError, MymathMissingError):
        print("⏳ ta librairie : sample_mvn n'est pas encore écrite (M25a.21)")
    except AssertionError as exc:
        print(f"❌ ta librairie échoue à ton test ({str(exc)[:200]}) : si ton test passe sur la référence, l'erreur est "
              "dans ta sample_mvn (lance aussi ses tests, M25a.21)")
    verdict("M25a.29", all(results_29), "ton test passe sur la référence et attrape les versions fausses.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : le nombre d'échecs de b) est-il compatible avec la borne de a) (300 répétitions laissent une fluctuation d'environ $\pm 0{,}6$ %) ? Pourquoi `sampler_diag` passe-t-il les comparaisons des moyennes et des variances ? Que dit e) de la taille d'échantillon qu'il faut pour voir une erreur fine ?

## Partie D · Défi

Fiche §4 et §5. Des températures manquantes, reconstruites par une loi conditionnelle.

### Ex M25a.30 — Défi : reconstruire les températures manquantes le long d'une route par la loi conditionnelle d'une gaussienne, sous une erreur quadratique moyenne fixée 🏆 ★★★ ⏱️ 30 min
**Objectif :** reconstruire des valeurs manquantes par la loi conditionnelle d'une gaussienne, en construisant soi-même la covariance qui tient compte du bruit de mesure.  
**Prérequis :** fiche §4 et §5 ; Ex M25a.22 · **Parcours :** C

Une route est équipée d'un capteur de température par kilomètre, aux positions $s = 0, 1, \ldots, 30$ km (`S_30`). Les vraies températures (en °C) suivent une loi normale de moyenne $m(s) = 14 - 0{,}15\,s$ (la route monte ; `MEAN_30`) et de covariance $K_{ij} = 4\exp\big(-\frac{\lvert s_i - s_j\rvert}{6}\big)$, exponentielle de la distance (`K_30`). Douze capteurs sont en panne, aux positions 0, 1, 2, 13 à 17, et 27 à 30 (`MISS_30`) ; les 19 autres (`OBS_30`) mesurent la vraie température avec une erreur gaussienne indépendante d'écart-type 0,4 °C (`NOISE_SD_30`). La cellule fournie tire les vraies températures et les mesures avec `rng = np.random.default_rng(5930)` (les vraies températures $m + \mathbf{L}\,$`rng.standard_normal(31)`, $\mathbf{L}$ étant le facteur de Cholesky de $\mathbf{K}$, puis les 19 mesures, vraies températures plus $0{,}4\,$`rng.standard_normal(19)`) ; elle ne te donne que les mesures (`Y_OBS_30`), et la cellule de vérification garde les vraies températures des positions en panne.

Écris `fill_missing_30(s, y_obs, obs, miss)`, qui rend les moyennes et les écarts-types de la loi conditionnelle des vraies températures aux positions `miss` sachant les mesures aux positions `obs` (deux tableaux de 12 nombres). Le bruit ne touche que les positions mesurées : à toi de construire la covariance du vecteur formé des vraies températures aux positions en panne et des mesures aux autres positions (ta `condition_gaussian` peut faire le reste). Écris aussi `draw_road_30(ax, s, y_obs, obs, miss, means, sds)`, qui trace dans l'axe reçu les mesures, tes moyennes avec leur bande à deux écarts-types, la moyenne $m(s)$ et l'interpolation linéaire `np.interp(s, s[obs], y_obs)`.

**Critère de réussite :** la cellule de vérification calcule l'erreur quadratique moyenne de tes douze moyennes contre les vraies températures, et réussit si elle est au plus 1,0 °C. Puis relève, dans ta copie : a) ton erreur quadratique moyenne (affichée par la vérification) ; b) celle de l'interpolation linéaire des mesures, `np.interp(s[miss], s[obs], y_obs)` (affichée aussi) ; c) combien des douze vraies températures tombent à moins de deux de tes écarts-types de tes moyennes (affiché aussi).

Pour aller plus loin : la longueur 6 km de la covariance est supposée connue ; trace, en fonction de cette longueur, le logarithme de l'évidence des 19 mesures (ta `mvn_logpdf`), et regarde où il est maximal.

In [ ]:
S_30 = np.arange(31.0)                                     # the positions of the sensors (km)
MEAN_30 = 14 - 0.15 * S_30                                 # the mean temperature m(s) (°C): the road climbs
K_30 = 4 * np.exp(-np.abs(S_30[:, None] - S_30[None, :]) / 6)    # K_ij = 4 exp(−|s_i − s_j| / 6)
MISS_30 = [0, 1, 2, 13, 14, 15, 16, 17, 27, 28, 29, 30]    # the twelve sensors down
OBS_30 = [i for i in range(31) if i not in MISS_30]        # the 19 sensors that measure
NOISE_SD_30 = 0.4                                          # the standard deviation of a measurement error (°C)


def measures_30():
    """The 19 measures at the positions OBS_30; the true temperatures stay inside this function."""
    rng = np.random.default_rng(5930)
    truth = MEAN_30 + np.linalg.cholesky(K_30) @ rng.standard_normal(31)
    return truth[OBS_30] + NOISE_SD_30 * rng.standard_normal(len(OBS_30))


Y_OBS_30 = measures_30()

In [ ]:
def fill_missing_30(s, y_obs, obs, miss):
    """The means and the standard deviations (two 1-D arrays of len(miss)) of the conditional law of the true
    temperatures at the positions miss, given the measures y_obs at the positions obs (model: mean 14 − 0.15 s,
    covariance 4 exp(−|s_i − s_j| / 6), a measurement noise of standard deviation 0.4 on the measured positions only)."""
    raise NotImplementedError("fill_missing_30 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_road_30(ax, s, y_obs, obs, miss, means, sds):
    """In ax: the measures (points), your means at the positions miss with their band ± 2 sds, the mean m(s) and the
    linear interpolation np.interp(s, s[obs], y_obs); labels and a legend. No other axes."""
    raise NotImplementedError("draw_road_30 n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
reconstructed_30 = None
with checked("M25a.30"):
    result_c30 = fill_missing_30(S_30.copy(), Y_OBS_30.copy(), list(OBS_30), list(MISS_30))
    means_c30, sds_c30 = pair("M25a.30", "fill_missing_30", result_c30, names=("means", "sds"))
    try:
        means_c30, sds_c30 = np.asarray(means_c30, dtype=float), np.asarray(sds_c30, dtype=float)
    except (TypeError, ValueError):
        verdict("M25a.30", False, "", "fill_missing_30 doit rendre deux tableaux de 12 nombres (moyennes, écarts-types).")
        raise StopCheck from None
    if means_c30.shape != (12,) or sds_c30.shape != (12,) or not (np.all(np.isfinite(means_c30))
                                                                  and np.all(np.isfinite(sds_c30))):
        verdict("M25a.30", False, "", f"fill_missing_30 doit rendre deux tableaux de 12 nombres finis, un par position "
                f"en panne ; reçu les formes {means_c30.shape} et {sds_c30.shape}.")
        raise StopCheck
    rng_c30 = np.random.default_rng(5930)
    truth_c30 = (MEAN_30 + np.linalg.cholesky(K_30) @ rng_c30.standard_normal(31))[MISS_30]   # kept by the check
    rmse_c30 = float(np.sqrt(np.mean((means_c30 - truth_c30) ** 2)))
    linear_c30 = np.interp(S_30[MISS_30], S_30[OBS_30], Y_OBS_30)
    rmse_linear_c30 = float(np.sqrt(np.mean((linear_c30 - truth_c30) ** 2)))
    inside_c30 = int(np.sum(np.abs(means_c30 - truth_c30) <= 2 * sds_c30))
    print(f"a) ton erreur quadratique moyenne : {fr(rmse_c30)} °C ; b) celle de l'interpolation linéaire : "
          f"{fr(rmse_linear_c30)} °C ; c) vraies températures à moins de deux de tes écarts-types : {inside_c30} sur 12")
    verdict("M25a.30", rmse_c30 <= 1.0, f"critère atteint : erreur quadratique moyenne {fr(rmse_c30)} °C ≤ 1,0 °C.",
            f"critère pas encore atteint : erreur quadratique moyenne {fr(rmse_c30)} °C > 1,0 °C.")
    reconstructed_30 = (means_c30, sds_c30)

with checked("M25a.30"):
    if reconstructed_30 is None:
        print("⏳ Ex M25a.30 : la figure attend une reconstruction qui passe la vérification ci-dessus.")
        raise StopCheck
    fig_30, ax_30 = plt.subplots(figsize=(10, 4.5))
    try:
        opened_30 = drawn(draw_road_30, ax_30, S_30, Y_OBS_30, list(OBS_30), list(MISS_30), *reconstructed_30)
    except BaseException:
        plt.close(fig_30)
        raise
    if opened_30:
        print("❌ Ex M25a.30 : draw_road_30 doit dessiner dans l'axe reçu, sans ouvrir de figure.")
    plt.show()

**Dans ta copie** : a), b) et c), lus sur la vérification ; où ta reconstruction gagne-t-elle sur l'interpolation linéaire, et pourquoi ?

**Pour réfléchir** : pourquoi la loi conditionnelle revient-elle vers la tendance $m(s)$ aux deux bouts de la route, alors que l'interpolation linéaire prolonge la dernière mesure ? Que deviendraient tes moyennes si tu oubliais le bruit de mesure ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu évaluer la densité et la log-densité d'une loi normale multivariée par Cholesky, et y lire le rôle de $\det\boldsymbol{\Sigma}$ et de la distance de Mahalanobis ?
2. Sais-tu trouver la loi d'une transformation affine, et tirer des vecteurs de loi $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$, en colonnes comme en lignes ?
3. Sais-tu tracer les ellipses de niveau, calculer leur masse en dimension 2, et dire ce qu'elle devient en dimension $d$ ?
4. Sais-tu calculer un complément de Schur et une loi conditionnelle, et expliquer pourquoi la covariance conditionnelle ne dépend pas de la valeur lue ?
5. Sais-tu fusionner des mesures, faire une régression bayésienne, calculer une KL entre deux gaussiennes, et écrire un test statistique dont tu connais le risque d'échec ?

**La suite** : M25b, qui introduit les variables cachées et réutilise ce chapitre (le conditionnement, la KL et l'entropie, le maximum de vraisemblance), puis le chapitre 25 de workbookIA, qui construit des modèles génératifs à partir de tirages gaussiens.